## 얼굴 복원 후처리 실험

조합 3 최종본에 복원 모델을 걸어 눈·코·입 형태가 개선되는지 본다.

입력은 upper 세트 6장의 비율 팽창 최종본이다. 팽창 20px 은 작은 얼굴에서
과하게 작용해 마스크를 깎는 것이 확인돼 비율 0.077 로 바꾼 결과를 쓴다.

후보는 CodeFormer·GFPGAN·GPEN·RestoreFormer++ 4종이다. 복원 모델은 선명화
필터가 아니라 실제 사람 얼굴을 학습한 생성 prior 로, 입력 얼굴을 사람 얼굴
분포 쪽으로 끌어당긴다.

주 판정은 눈·코·입 확대 육안이다. 과복원으로 매끈해지는 것이 "로봇 같음"
지적과 같은 방향이라 라플라시안 분산을 원본 실측 63.4 와 대조해 참고한다.
조합 3 은 참조 얼굴을 닮는 것이 목적이므로 InsightFace 코사인으로 이탈을 본다.

In [ ]:
import subprocess

import torch

print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    ).stdout
)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

from google.colab import drive

drive.mount("/content/drive")

!apt-get install -qq fonts-nanum > /dev/null
!fc-cache -fv > /dev/null

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

BASE = "/content/drive/MyDrive/saloncut_data"
IN_DIR = f"{BASE}/outputs/upper_combo3/dilate_ratio"
!ls {IN_DIR}

## CodeFormer 설치

basicsr 계열 의존성을 쓴다. 코랩에서는 검증된 경로가 있으나 최신 환경에서
torchvision API 변경으로 import 가 깨지는 사례가 알려져 있어, 설치 후
import 까지 확인한다.

가중치는 첫 실행 시 자동으로 받는다.

In [ ]:
%cd /content
!git clone -q https://github.com/sczhou/CodeFormer.git
%cd /content/CodeFormer

!pip install -q -r requirements.txt 2>&1 | tail -3
!python basicsr/setup.py develop 2>&1 | tail -3

# 가중치 확보
!python scripts/download_pretrained_models.py facelib 2>&1 | tail -2
!python scripts/download_pretrained_models.py CodeFormer 2>&1 | tail -2

import sys

sys.path.insert(0, "/content/CodeFormer")

from basicsr.utils.registry import ARCH_REGISTRY

print("import 성공")

## CodeFormer fidelity 스윕

fidelity 가중치 w 는 복원 강도와 원본 충실도의 균형이다.

  w 낮음  prior 우세. 사람 얼굴 분포로 강하게 끌어당김. 구조 교정이 크지만
          원본에서 멀어지고 피부가 매끈해진다
  w 높음  입력 충실. 원본 유지. 구조 개선은 작다

"실제 사람 같지 않다"가 문제이므로 낮은 w 쪽이 유력하다. 0.3·0.5·0.7·0.9 를 돌린다.

--bg_upsampler 는 쓰지 않는다. 배경까지 손대면 우리가 지킨 헤어·배경이 바뀐다.
--face_upsample 도 끈다. 크기가 달라지면 비교가 안 된다.

In [ ]:
import time
from pathlib import Path

IN_DIR = Path("/content/drive/MyDrive/saloncut_data/outputs/upper_combo3/dilate_ratio")
CF_OUT = Path("/content/cf")
CF_OUT.mkdir(exist_ok=True)

WEIGHTS = (0.3, 0.5, 0.7, 0.9)

%cd /content/CodeFormer
for w in WEIGHTS:
    t0 = time.time()
    !python inference_codeformer.py -w {w} -i {IN_DIR} -o {CF_OUT}/w{w} 2>&1 | tail -2
    print(f"w={w}  {time.time() - t0:.1f}초")

!find {CF_OUT} -name "*.png" | head -30

## fidelity 스윕 비교

cropped_faces 가 복원 전 512 정렬 얼굴, restored_faces 가 복원 후다.
같은 좌표계라 나란히 놓으면 차이가 그대로 보인다.

주 판정은 눈·코·입 형태다. 과복원으로 매끈해지는지도 같이 본다.

In [ ]:
from PIL import Image

CF = Path("/content/cf")
NAMES = [
    "c3_upper_06_asian_male",
    "c3_upper_05_asian_glasses",
    "c3_upper_04_asian_landscape",
    "c3_upper_02_west_small_face",
    "c3_upper_03_asian_tied",
    "c3_upper_01_asian_long_dark",
]
LAT = [2.3, 2.5, 3.2, 3.3, 4.7, 5.3]

FIG = Path("/content/drive/MyDrive/saloncut_data/outputs/report_figures")


def savefig(num, name):
    p = FIG / f"fig{num}_{name}.png"
    plt.savefig(p, dpi=120, bbox_inches="tight")
    print(f"저장  {p.name}")


rows_lab = ["복원 전"] + [f"w={w}" for w in WEIGHTS]

fig, axes = plt.subplots(5, 6, figsize=(22, 19))
for c, (name, lat) in enumerate(zip(NAMES, LAT)):
    srcs = [CF / "w0.3" / "cropped_faces" / f"{name}_00.png"] + [
        CF / f"w{w}" / "restored_faces" / f"{name}_00.png" for w in WEIGHTS
    ]
    for r, (lab, p) in enumerate(zip(rows_lab, srcs)):
        axes[r, c].imshow(Image.open(p))
        axes[r, c].set_title(f"{lab}  {name[9:19]}  L{lat}", fontsize=9)
        axes[r, c].axis("off")

plt.tight_layout()
savefig(90, "codeformer_fidelity_sweep")
plt.show()

## 전체 결과 비교

지금까지는 512 정렬 얼굴만 봤다. 실제 사용자가 보는 것은 원본에 다시 붙인
final_results 다. 얼굴이 헤어·몸과 어울리는지, 붙인 경계가 드러나는지는
전체에서만 확인된다.

업로드 원본 · 복원 전 · w0.3 · w0.9 를 나란히 둔다. w 중간값은 앞선 확대에서
0.3 과 0.9 사이에 놓이는 것이 확인됐으므로 양 끝만 본다.

In [ ]:
UPPER = Path("/content/drive/MyDrive/saloncut_data/test_images/upper")

LAB4 = ("업로드 원본", "복원 전", "w=0.3", "w=0.9")

fig, axes = plt.subplots(4, 6, figsize=(22, 22))
for c, (name, lat) in enumerate(zip(NAMES, LAT)):
    stem = name[3:]  # c3_ 제거
    srcs = (
        UPPER / f"{stem}.jpg",
        IN_DIR / f"{name}.png",
        CF / "w0.3" / "final_results" / f"{name}.png",
        CF / "w0.9" / "final_results" / f"{name}.png",
    )
    for r, (lab, p) in enumerate(zip(LAB4, srcs)):
        axes[r, c].imshow(Image.open(p))
        axes[r, c].set_title(f"{lab}  {stem[6:16]}  L{lat}", fontsize=9)
        axes[r, c].axis("off")

plt.tight_layout()
savefig(91, "codeformer_full_compare")
plt.show()

In [ ]:
import shutil

CF_DRIVE = Path("/content/drive/MyDrive/saloncut_data/outputs/upper_restore/B_after")
CF_DRIVE.mkdir(parents=True, exist_ok=True)

for w in WEIGHTS:
    for sub in ("final_results", "restored_faces", "cropped_faces"):
        dst = CF_DRIVE / f"w{w}" / sub
        dst.mkdir(parents=True, exist_ok=True)
        for p in (CF / f"w{w}" / sub).glob("*.png"):
            shutil.copy(p, dst / p.name)

print(f"저장  {CF_DRIVE}")
!ls {CF_DRIVE}

## A 방식 순서 실험

현재는 복원이 마지막이라 색 정합으로 맞춘 톤을 복원이 덮는다. upper_02 의
톤 단절이 여기서 나온 것으로 보인다.

복원을 색 정합 앞으로 옮기면 복원이 형태만 고치고 톤은 색 정합이 원본에
맞추게 된다. 각 단계가 제 역할만 하는 구조다.

  B (현재)  생성 → 색정합 → 재합성 → 고주파 → 복원
  A (제안)  생성 → 복원 → 색정합 → 재합성 → 고주파

한 세션에 SDXL 과 CodeFormer 를 모두 올려야 하므로 의존성 공존부터 확인한다.

In [ ]:
import subprocess

import torch

print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    ).stdout
)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

from google.colab import drive

drive.mount("/content/drive")

!apt-get install -qq fonts-nanum > /dev/null
!fc-cache -fv > /dev/null

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

# --- CodeFormer ---

%cd /content
!git clone -q https://github.com/sczhou/CodeFormer.git
%cd /content/CodeFormer
!pip install -q -r requirements.txt 2>&1 | tail -3
!python basicsr/setup.py develop 2>&1 | tail -2
!python scripts/download_pretrained_models.py facelib 2>&1 | tail -1
!python scripts/download_pretrained_models.py CodeFormer 2>&1 | tail -1

# --- SalonCutAI ---

%cd /content
!git clone -q https://github.com/qja0707/SalonCutAI.git
%cd /content/SalonCutAI/backend
!pip install -q diffusers transformers accelerate insightface onnxruntime-gpu mediapipe opencv-python 2>&1 | tail -3

import os
import sys

os.environ["SALON_STORAGE_DIR"] = "/content/storage"
os.environ["IMAGE_GEN_ENABLED"] = "1"
sys.path.insert(0, "/content/SalonCutAI/backend")
sys.path.insert(0, "/content/CodeFormer")

# --- 공존 확인 ---

import numpy as np

from basicsr.utils.registry import ARCH_REGISTRY
from src.ai_engine.image_gen import combo3, compose, downloads, loader, masks

print("numpy", np.__version__, "| torch", torch.__version__)
print("공존 확인 완료")

downloads.ensure_models()
print("모델 준비 완료")

In [ ]:
%cd /content/CodeFormer

# CodeFormer setup.py 가 Python 3.12 에서 버전을 못 읽는다. 그 부분만 고친다.
import pathlib

p = pathlib.Path("/content/CodeFormer/basicsr/setup.py")
s = p.read_text()
old = "        exec(compile(f.read(), version_file, 'exec'))\n    return locals()['__version__']"
new = "        ns = {}\n        exec(compile(f.read(), version_file, 'exec'), ns)\n    return ns['__version__']"

if old in s:
    p.write_text(s.replace(old, new))
    print("패치 완료")
else:
    print("패치 대상 없음 — 코드가 다르다")

In [ ]:
%cd /content/CodeFormer
!python basicsr/setup.py develop 2>&1 | tail -5
!python scripts/download_pretrained_models.py facelib 2>&1 | tail -1
!python scripts/download_pretrained_models.py CodeFormer 2>&1 | tail -1

import basicsr
print("basicsr", basicsr.__version__, "|", basicsr.__file__)

In [ ]:
from pathlib import Path

IN_DIR = Path("/content/drive/MyDrive/saloncut_data/outputs/upper_combo3/dilate_ratio")
TEST_OUT = Path("/content/cf_test")

%cd /content/CodeFormer
!python inference_codeformer.py -w 0.3 -i {IN_DIR}/c3_upper_06_asian_male.png -o {TEST_OUT} 2>&1 | tail -5

!find {TEST_OUT} -name "*.png"

## A 방식 — 1단계 생성

조합 3 생성 결과를 저장한다. 복원이 CLI 스크립트라 파일로 주고받는다.

seed 42, 참조는 여성 5장 ref-01, 남성 1장 ref-05 로 B 방식과 동일하게 맞춘다.
후처리에 쓸 1024 축소 원본도 함께 저장한다.

In [ ]:
import time

import numpy as np
from PIL import Image

from src.ai_engine.image_gen import combo3, compose, loader as gen_loader, masks

BASE = Path("/content/drive/MyDrive/saloncut_data")
UPPER = BASE / "test_images/upper"
REF = BASE / "ref_faces"

GEN_DIR = Path("/content/a_gen")
SRC_DIR = Path("/content/a_src")
for d in (GEN_DIR, SRC_DIR):
    d.mkdir(exist_ok=True)

SEED = 42
DILATE_RATIO = 0.077
REF_MAP = {"upper_06_asian_male": "ref-05"}

files = sorted(UPPER.glob("*.jpg"))

for path in files:
    ref_id = REF_MAP.get(path.stem, "ref-01")
    t0 = time.time()

    out, img_r, _ = combo3.generate(
        Image.open(path).convert("RGB"), REF / f"{ref_id}.png", SEED
    )

    out.save(GEN_DIR / f"c3_{path.stem}.png")
    img_r.save(SRC_DIR / f"c3_{path.stem}.png")
    print(f"{path.stem:<26} {time.time() - t0:5.1f}초")

print("\n생성 완료")
!ls {GEN_DIR}

## A 방식 — 2단계 복원

생성 직후에 복원을 건다. 아직 색 정합·재합성 전이라 이 단계에서 톤이 바뀌어도
뒤따르는 색 정합이 원본 톤으로 되돌린다.

B 방식에서 w 0.3 이 6장 중 5장에서 가장 나았으므로 같은 값을 쓴다. 순서가
유일한 변수여야 비교가 성립한다.

In [ ]:
W = 0.3
REST_DIR = Path("/content/a_rest")

%cd /content/CodeFormer
!python inference_codeformer.py -w {W} -i {GEN_DIR} -o {REST_DIR} 2>&1 | tail -3

%cd /content/SalonCutAI/backend
!ls {REST_DIR}/final_results

## A 방식 — 3단계 후처리

복원본에 색 정합·재합성·고주파를 건다. 마스크는 1024 축소 원본에서 뽑고,
헤어 팽창은 검증이 끝난 얼굴 폭 비율 0.077 을 쓴다.

B 방식 최종본과 원본 코사인을 나란히 재서 순서 변경이 회피에 영향을 주는지 본다.

In [ ]:
A_FINAL = Path("/content/drive/MyDrive/saloncut_data/outputs/upper_restore/A_before")
A_FINAL.mkdir(parents=True, exist_ok=True)

B_FINAL = Path("/content/drive/MyDrive/saloncut_data/outputs/upper_restore/B_after/w0.3/final_results")


def identity(img_a, img_b):
    app = gen_loader.get_face_app()
    embs = []
    for im in (img_a, img_b):
        f = app.get(np.array(im)[:, :, ::-1])
        if not f:
            return None
        embs.append(f[0].normed_embedding)
    return float(np.dot(embs[0], embs[1]))


rows_a = []
for path in files:
    name = f"c3_{path.stem}"
    img_r = Image.open(SRC_DIR / f"{name}.png").convert("RGB")
    rest = Image.open(REST_DIR / "final_results" / f"{name}.png").convert("RGB")

    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]

    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    b = compose.color_transfer(rest.resize(img_r.size), img_r, gen_mask)
    c = compose.align_then_recompose(img_r, b, face_mask, hair_mask)
    d = compose.transfer_high_freq(c, img_r, gen_mask)

    d.save(A_FINAL / f"{name}.png")

    b_img = Image.open(B_FINAL / f"{name}.png").convert("RGB")
    rows_a.append((path.stem, identity(img_r, d), identity(img_r, b_img.resize(img_r.size))))
    print(f"{path.stem:<26} 완료")

print(f"\n{'file':<26}{'A 코사인':>11}{'B 코사인':>11}{'차이':>10}")
for n, ca, cb in rows_a:
    print(f"{n:<26}{ca:>11.4f}{cb:>11.4f}{ca - cb:>+10.4f}")

In [ ]:
FIG = Path("/content/drive/MyDrive/saloncut_data/outputs/report_figures")


def savefig(num, name):
    p = FIG / f"fig{num}_{name}.png"
    plt.savefig(p, dpi=120, bbox_inches="tight")
    print(f"저장  {p.name}")


ORDER = [
    "upper_06_asian_male", "upper_05_asian_glasses", "upper_04_asian_landscape",
    "upper_02_west_small_face", "upper_03_asian_tied", "upper_01_asian_long_dark",
]
LAT = {"upper_06_asian_male": 2.3, "upper_05_asian_glasses": 2.5,
       "upper_04_asian_landscape": 3.2, "upper_02_west_small_face": 3.3,
       "upper_03_asian_tied": 4.7, "upper_01_asian_long_dark": 5.3}

LAB = ("업로드 원본", "B 복원 마지막", "A 복원 먼저")

fig, axes = plt.subplots(3, 6, figsize=(22, 17))
for c, name in enumerate(ORDER):
    srcs = (
        UPPER / f"{name}.jpg",
        B_FINAL / f"c3_{name}.png",
        A_FINAL / f"c3_{name}.png",
    )
    for r, (lab, p) in enumerate(zip(LAB, srcs)):
        axes[r, c].imshow(Image.open(p))
        axes[r, c].set_title(f"{lab}  {name[6:16]}  L{LAT[name]}", fontsize=9)
        axes[r, c].axis("off")

plt.tight_layout()
savefig(92, "restore_order_AB_compare")
plt.show()

## B2 방식 — 복원 후 색 정합 추가

A 방식은 기각한다. 콧대 띠는 어파인 정렬 이동으로 생기는데, A 는 복원 뒤에
정렬이 와서 띠를 덮을 것이 없다. B 에서 띠가 사라졌던 것은 복원이 마지막이라
띠를 덮었기 때문이다.

B 를 유지하면서 톤만 잡는다. 복원 뒤에 색 정합을 한 번 더 걸어 얼굴 톤을
원본에 맞춘다. 색 정합은 색만 옮기고 형태는 건드리지 않으므로 띠가 다시
생기지 않는다. 앞선 측정에서 색 정합의 코사인 기여는 ±0.01 로 거의 0 이었다.

  B    생성 → 색정합 → 재합성 → 고주파 → 복원
  B2   생성 → 색정합 → 재합성 → 고주파 → 복원 → 색정합

In [ ]:
B2_DIR = Path("/content/drive/MyDrive/saloncut_data/outputs/upper_restore/B2_recolor")
B2_DIR.mkdir(parents=True, exist_ok=True)

rows_b2 = []
for path in files:
    name = f"c3_{path.stem}"
    img_r = Image.open(SRC_DIR / f"{name}.png").convert("RGB")
    b_img = Image.open(B_FINAL / f"{name}.png").convert("RGB").resize(img_r.size)

    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]

    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    b2 = compose.color_transfer(b_img, img_r, gen_mask)
    b2.save(B2_DIR / f"{name}.png")

    rows_b2.append((path.stem, identity(img_r, b_img), identity(img_r, b2)))
    print(f"{path.stem:<26} 완료")

print(f"\n{'file':<26}{'B':>10}{'B2':>10}{'변화':>10}")
for n, cb, c2 in rows_b2:
    print(f"{n:<26}{cb:>10.4f}{c2:>10.4f}{c2 - cb:>+10.4f}")

In [ ]:
LAB3 = ("업로드 원본", "B", "B2 색정합 추가")

fig, axes = plt.subplots(3, 6, figsize=(22, 17))
for c, name in enumerate(ORDER):
    srcs = (
        UPPER / f"{name}.jpg",
        B_FINAL / f"c3_{name}.png",
        B2_DIR / f"c3_{name}.png",
    )
    for r, (lab, p) in enumerate(zip(LAB3, srcs)):
        axes[r, c].imshow(Image.open(p))
        axes[r, c].set_title(f"{lab}  {name[6:16]}  L{LAT[name]}", fontsize=9)
        axes[r, c].axis("off")

plt.tight_layout()
savefig(93, "restore_B2_recolor_compare")
plt.show()

In [ ]:
import cv2


def to_lab(img):
    return cv2.cvtColor(np.array(img).astype(np.float32) / 255, cv2.COLOR_RGB2LAB)


def delta_e(x, y):
    return np.linalg.norm(to_lab(x) - to_lab(y), axis=2)


print("정의 완료")

In [ ]:
print(f"{'file':<26}{'마스크안 ΔE':>13}{'마스크밖 ΔE':>13}")
for path in files:
    name = f"c3_{path.stem}"
    img_r = Image.open(SRC_DIR / f"{name}.png").convert("RGB")
    b_img = Image.open(B_FINAL / f"{name}.png").convert("RGB").resize(img_r.size)
    b2 = Image.open(B2_DIR / f"{name}.png").convert("RGB")

    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]
    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
    m = np.array(masks.build_gen_mask(face_mask, hair_mask)) > 127

    de = delta_e(b_img, b2)
    print(f"{path.stem:<26}{de[m].mean():>13.2f}{de[~m].mean():>13.2f}")

## B2 수정 — 마스크 안만 채택

color_transfer 는 마스크로 색 통계만 뽑고 변환은 이미지 전체에 적용한다.
파이프라인에서는 뒤따르는 재합성이 마스크 밖을 원본으로 덮어 문제가 없었으나,
B2 는 색 정합이 마지막이라 배경까지 바뀐다. 마스크 밖 ΔE 가 안쪽보다 크게
나온 것이 그 결과다(upper_02 기준 9.24 대 2.30).

색 정합 결과에서 마스크 안만 취하고 밖은 복원본을 그대로 둔다.

In [ ]:
B2_DIR = Path("/content/drive/MyDrive/saloncut_data/outputs/upper_restore/B2_recolor")

rows_b2 = []
for path in files:
    name = f"c3_{path.stem}"
    img_r = Image.open(SRC_DIR / f"{name}.png").convert("RGB")
    b_img = Image.open(B_FINAL / f"{name}.png").convert("RGB").resize(img_r.size)

    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]
    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    shifted = compose.color_transfer(b_img, img_r, gen_mask)
    b2 = Image.composite(shifted, b_img, gen_mask.convert("L"))
    b2.save(B2_DIR / f"{name}.png")

    m = np.array(gen_mask) > 127
    de = delta_e(b_img, b2)
    rows_b2.append((path.stem, de[m].mean(), de[~m].mean(), identity(img_r, b2)))

print(f"{'file':<26}{'마스크안':>10}{'마스크밖':>10}{'코사인':>10}")
for n, di, do, ci in rows_b2:
    print(f"{n:<26}{di:>10.2f}{do:>10.2f}{ci:>10.4f}")

In [ ]:
LAB3 = ("업로드 원본", "B", "B2 마스크 적용")

fig, axes = plt.subplots(3, 6, figsize=(22, 17))
for c, name in enumerate(ORDER):
    srcs = (
        UPPER / f"{name}.jpg",
        B_FINAL / f"c3_{name}.png",
        B2_DIR / f"c3_{name}.png",
    )
    for r, (lab, p) in enumerate(zip(LAB3, srcs)):
        axes[r, c].imshow(Image.open(p))
        axes[r, c].set_title(f"{lab}  {name[6:16]}  L{LAT[name]}", fontsize=9)
        axes[r, c].axis("off")

plt.tight_layout()
savefig(93, "restore_B2_recolor_compare")
plt.show()

In [ ]:
def face_crop(img, pad=1.5, size=420):
    det = gen_loader.get_face_app().get(np.array(img)[:, :, ::-1])
    if not det:
        return None
    x1, y1, x2, y2 = det[0].bbox
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    half = max(x2 - x1, y2 - y1) / 2 * pad
    box = (
        max(0, int(cx - half)), max(0, int(cy - half)),
        min(img.width, int(cx + half)), min(img.height, int(cy + half)),
    )
    c = img.crop(box)
    return c.resize((size, int(size * c.height / c.width)), Image.LANCZOS)


fig, axes = plt.subplots(3, 6, figsize=(22, 12))
for c, name in enumerate(ORDER):
    img_r = Image.open(SRC_DIR / f"c3_{name}.png").convert("RGB")
    srcs = (
        img_r,
        Image.open(B_FINAL / f"c3_{name}.png").convert("RGB").resize(img_r.size),
        Image.open(B2_DIR / f"c3_{name}.png").convert("RGB"),
    )
    for r, (lab, im) in enumerate(zip(LAB3, srcs)):
        cr = face_crop(im)
        axes[r, c].imshow(cr if cr else Image.new("RGB", (420, 420)))
        axes[r, c].set_title(f"{lab}  {name[6:16]}  L{LAT[name]}", fontsize=9)
        axes[r, c].axis("off")

plt.tight_layout()
savefig(94, "restore_B2_face_zoom")
plt.show()

In [ ]:
DR = Path("/content/drive/MyDrive/saloncut_data/outputs/upper_restore")
!ls -R {DR} | head -40

In [ ]:
STAGE_DIR = Path("/content/drive/MyDrive/saloncut_data/outputs/upper_combo3/stages_ratio")
name = "upper_02_west_small_face"

srcs = [
    (SRC_DIR / f"c3_{name}.png", "1024 원본"),
    (STAGE_DIR / f"{name}_A_생성.png", "생성"),
    (STAGE_DIR / f"{name}_C_재합성.png", "재합성"),
    (STAGE_DIR / f"{name}_D_고주파.png", "고주파"),
    (B2_DIR / f"c3_{name}.png", "복원+색정합"),
]

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
for ax, (p, lab) in zip(axes, srcs):
    ax.imshow(face_crop(Image.open(p).convert("RGB"), pad=1.1, size=400))
    ax.set_title(lab, fontsize=11)
    ax.axis("off")
plt.tight_layout()
savefig(95, "upper02_eye_stage_trace")
plt.show()

## upper_02 이목구비 몰림 원인 확인

생성 단계에서 이미 얼굴이 작아지고 이목구비가 중앙으로 몰린다. 8/12 인페인팅
모델 비교에서 SD 1.5 계열에 나타났던 것과 같은 실패다.

원인 후보가 둘이다.
  얼굴 크기  132px 로 작아 InstantID 키포인트 정밀도가 떨어진다
  인종 불일치  서양인 골격에 한국인 참조를 씌워 눈 간격·코 높이가 안 맞는다

서양인 참조로 바꿔 생성한다. 몰림이 사라지면 인종 문제, 그대로면 크기 문제다.

In [ ]:
name = "upper_02_west_small_face"
src = Image.open(UPPER / f"{name}.jpg").convert("RGB")

CANDS = ("ref-01", "ref-23")  # 한국 20 여 / 외국 여 서양

outs = {}
for r in CANDS:
    out, img_r, _ = combo3.generate(src, REF / f"{r}.png", SEED)
    outs[r] = out
    print(f"{r} 완료")

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
panels = [
    (Image.open(REF / "ref-01.png").convert("RGB"), "참조 ref-01"),
    (Image.open(REF / "ref-23.png").convert("RGB"), "참조 ref-23"),
    (img_r, "1024 원본"),
    (outs["ref-01"], "생성 ref-01"),
    (outs["ref-23"], "생성 ref-23"),
]
for ax, (im, lab) in zip(axes, panels):
    c = face_crop(im, pad=1.2, size=400)
    ax.imshow(c if c else im)
    ax.set_title(lab, fontsize=11)
    ax.axis("off")

plt.tight_layout()
savefig(96, "upper02_ref_race_test")
plt.show()

In [ ]:
name = "upper_02_west_small_face"
STAGE_DIR = Path("/content/drive/MyDrive/saloncut_data/outputs/upper_combo3/stages_ratio")

srcs = [
    (SRC_DIR / f"c3_{name}.png", "1024 원본"),
    (STAGE_DIR / f"{name}_A_생성.png", "생성"),
    (STAGE_DIR / f"{name}_B_색정합.png", "색정합"),
    (STAGE_DIR / f"{name}_C_재합성.png", "재합성"),
    (STAGE_DIR / f"{name}_D_고주파.png", "고주파"),
    (B2_DIR / f"c3_{name}.png", "복원+색정합"),
]

fig, axes = plt.subplots(1, 6, figsize=(24, 5))
for ax, (p, lab) in zip(axes, srcs):
    ax.imshow(face_crop(Image.open(p).convert("RGB"), pad=1.2, size=400))
    ax.set_title(lab, fontsize=11)
    ax.axis("off")
plt.tight_layout()
savefig(97, "upper02_eye_stage_trace_v2")
plt.show()

In [ ]:
NOALIGN = Path("/content/drive/MyDrive/saloncut_data/outputs/upper_combo3/no_align")
name = "upper_02_west_small_face"

srcs = [
    (SRC_DIR / f"c3_{name}.png", "1024 원본"),
    (STAGE_DIR / f"{name}_A_생성.png", "생성"),
    (STAGE_DIR / f"{name}_D_고주파.png", "정렬 있음"),
    (NOALIGN / f"c3_{name}.png", "정렬 없음"),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (p, lab) in zip(axes, srcs):
    ax.imshow(face_crop(Image.open(p).convert("RGB"), pad=1.2, size=400))
    ax.set_title(lab, fontsize=11)
    ax.axis("off")
plt.tight_layout()
savefig(98, "upper02_align_eye_check")
plt.show()

In [ ]:
rows_kps = []
for path in files:
    name = f"c3_{path.stem}"
    img_r = Image.open(SRC_DIR / f"{name}.png").convert("RGB")
    gen = Image.open(STAGE_DIR / f"{path.stem}_A_생성.png").convert("RGB")

    app = gen_loader.get_face_app()
    fo = app.get(np.array(img_r)[:, :, ::-1])
    fg = app.get(np.array(gen)[:, :, ::-1])
    if not fo or not fg:
        rows_kps.append((path.stem, None, None, None))
        continue

    ko, kg = fo[0].kps, fg[0].kps
    fw = fo[0].bbox[2] - fo[0].bbox[0]
    d = np.linalg.norm(kg - ko, axis=1)

    rows_kps.append((path.stem, d.mean(), d.max(), fw))

print(f"{'file':<26}{'평균오차':>9}{'최대오차':>9}{'얼굴폭':>8}{'평균/폭':>9}")
for n, avg, mx, fw in rows_kps:
    if avg is None:
        print(f"{n:<26}{'검출실패':>9}")
        continue
    print(f"{n:<26}{avg:>9.1f}{mx:>9.1f}{fw:>8.0f}{avg / fw:>9.3f}")

In [ ]:
KP_NAMES = ("왼눈", "오른눈", "코", "왼입", "오른입")

for path in files:
    name = f"c3_{path.stem}"
    img_r = Image.open(SRC_DIR / f"{name}.png").convert("RGB")
    gen = Image.open(STAGE_DIR / f"{path.stem}_A_생성.png").convert("RGB")

    app = gen_loader.get_face_app()
    fo, fg = app.get(np.array(img_r)[:, :, ::-1]), app.get(np.array(gen)[:, :, ::-1])
    if not fo or not fg:
        continue

    ko, kg = fo[0].kps, fg[0].kps
    d = kg - ko

    m, _ = cv2.estimateAffinePartial2D(kg.astype(np.float32), ko.astype(np.float32),
                                       method=cv2.LMEDS)
    tx, ty = m[0, 2], m[1, 2]

    print(f"\n{path.stem}  정렬 이동 ({tx:+.1f}, {ty:+.1f})")
    for n, v in zip(KP_NAMES, d):
        print(f"  {n:<5} ({v[0]:+6.1f}, {v[1]:+6.1f})")

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(24, 9))
for c, name in enumerate(ORDER):
    srcs = (
        (STAGE_DIR / f"{name}_A_생성.png", "생성"),
        (STAGE_DIR / f"{name}_C_재합성.png", "재합성"),
    )
    for r, (p, lab) in enumerate(srcs):
        axes[r, c].imshow(face_crop(Image.open(p).convert("RGB"), pad=1.2, size=400))
        axes[r, c].set_title(f"{lab}  {name[6:16]}", fontsize=9)
        axes[r, c].axis("off")

plt.tight_layout()
savefig(100, "align_eye_shift_all")
plt.show()

## 정렬 방식 비교

kps 5점 오차가 1.4~4.4px 인데 현재 정렬은 5.3~26px 을 민다. 회전·스케일 성분이
평행이동으로 흘러들어간 것으로 보인다. estimateAffinePartial2D 는 4자유도를
푸는데 점이 5개뿐이라 미세한 오차에도 과적합한다.

세 방식을 비교한다.
  현재    유사변환 (이동·회전·스케일)
  이동만  5점 평균 차이만큼 평행이동
  없음    정렬 생략

생성 결과의 눈 위치가 유지되는지, 코사인이 얼마나 달라지는지 함께 본다.

In [ ]:
import cv2

from src.ai_engine.image_gen import compose as comp_mod

ALIGN_DIRS = {}
for tag in ("affine", "shift", "none"):
    d = Path(f"/content/drive/MyDrive/saloncut_data/outputs/upper_combo3/align_{tag}")
    d.mkdir(parents=True, exist_ok=True)
    ALIGN_DIRS[tag] = d


def recompose(mode, img_r, out, face_mask, hair_mask):
    """mode 에 따라 정렬 방식을 바꿔 재합성한다."""
    res = out.resize(img_r.size)
    o, g = np.array(img_r.convert("RGB")), np.array(res.convert("RGB"))

    if mode != "none":
        ko, kg = comp_mod._get_kps(o), comp_mod._get_kps(g)
        if ko is not None and kg is not None:
            if mode == "affine":
                m, _ = cv2.estimateAffinePartial2D(kg, ko, method=cv2.LMEDS)
            else:  # shift
                d = (ko - kg).mean(axis=0)
                m = np.array([[1.0, 0.0, d[0]], [0.0, 1.0, d[1]]])
            if m is not None:
                res = Image.fromarray(
                    cv2.warpAffine(g, m, (o.shape[1], o.shape[0]),
                                   flags=cv2.INTER_LANCZOS4)
                )

    return compose.recompose_with_hair(img_r, res, face_mask, hair_mask)


rows_al = []
for path in files:
    name = f"c3_{path.stem}"
    img_r = Image.open(SRC_DIR / f"{name}.png").convert("RGB")
    out = Image.open(STAGE_DIR / f"{path.stem}_A_생성.png").convert("RGB")

    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]
    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    b = compose.color_transfer(out, img_r, gen_mask)

    vals = []
    for tag in ("affine", "shift", "none"):
        c = recompose(tag, img_r, b, face_mask, hair_mask)
        d = compose.transfer_high_freq(c, img_r, gen_mask)
        d.save(ALIGN_DIRS[tag] / f"{name}.png")
        vals.append(identity(img_r, d))

    rows_al.append((path.stem, *vals))
    print(f"{path.stem:<26} 완료")

print(f"\n{'file':<26}{'현재':>10}{'이동만':>10}{'없음':>10}")
for n, a, s, no in rows_al:
    print(f"{n:<26}{a:>10.4f}{s:>10.4f}{no:>10.4f}")

In [ ]:
LABS = ("생성", "현재 유사변환", "이동만", "정렬 없음")

fig, axes = plt.subplots(4, 6, figsize=(24, 18))
for c, name in enumerate(ORDER):
    srcs = (
        STAGE_DIR / f"{name}_A_생성.png",
        ALIGN_DIRS["affine"] / f"c3_{name}.png",
        ALIGN_DIRS["shift"] / f"c3_{name}.png",
        ALIGN_DIRS["none"] / f"c3_{name}.png",
    )
    for r, (lab, p) in enumerate(zip(LABS, srcs)):
        axes[r, c].imshow(face_crop(Image.open(p).convert("RGB"), pad=1.2, size=400))
        axes[r, c].set_title(f"{lab}  {name[6:16]}", fontsize=9)
        axes[r, c].axis("off")

plt.tight_layout()
savefig(101, "align_3way_compare")
plt.show()

## normal 세트 정렬 필요성 검증

정렬은 "img2img 라 얼굴이 미세하게 어긋난다"는 이유로 들어갔다. 얼빡 세트에서
정한 것으로 보이는데, upper 세트에서는 kps 오차가 1.4~4.4px 로 무시할 수준이었다.

normal 10장에서 kps 오차와 정렬 이동량을 재고, 정렬 유무 결과를 비교한다.
얼빡 세트에서 정렬이 실제로 필요했다면 오차가 크게 나와야 한다.

In [ ]:
NORMAL = Path("/content/drive/MyDrive/saloncut_data/test_images/normal")
N_DIRS = {}
for tag in ("affine", "none"):
    d = Path(f"/content/drive/MyDrive/saloncut_data/outputs/normal_align/{tag}")
    d.mkdir(parents=True, exist_ok=True)
    N_DIRS[tag] = d

n_files = sorted(NORMAL.glob("*.jpg"))
rows_n = []

for path in n_files:
    ref_id = "ref-05" if "male" in path.stem else "ref-01"
    out, img_r, _ = combo3.generate(
        Image.open(path).convert("RGB"), REF / f"{ref_id}.png", SEED
    )

    o, g = np.array(img_r.convert("RGB")), np.array(out.resize(img_r.size).convert("RGB"))
    ko, kg = comp_mod._get_kps(o), comp_mod._get_kps(g)
    if ko is None or kg is None:
        rows_n.append((path.stem, None, None, None, None, None))
        continue

    err = np.linalg.norm(kg - ko, axis=1).mean()
    m, _ = cv2.estimateAffinePartial2D(kg, ko, method=cv2.LMEDS)
    shift = np.hypot(m[0, 2], m[1, 2])
    scale = np.hypot(m[0, 0], m[1, 0])

    det = gen_loader.get_face_app().get(o[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]

    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)
    b = compose.color_transfer(out, img_r, gen_mask)

    cos = []
    for tag in ("affine", "none"):
        c = recompose(tag, img_r, b, face_mask, hair_mask)
        d = compose.transfer_high_freq(c, img_r, gen_mask)
        d.save(N_DIRS[tag] / f"c3_{path.stem}.png")
        cos.append(identity(img_r, d))
    img_r.save(N_DIRS["none"] / f"src_{path.stem}.png")

    rows_n.append((path.stem, err, shift, scale, fw, cos))
    print(f"{path.stem:<26} 완료")

print(f"\n{'file':<26}{'kps오차':>9}{'정렬이동':>9}{'스케일':>8}{'얼굴폭':>8}{'현재':>9}{'없음':>9}")
for n, e, s, sc, fw, cos in rows_n:
    if e is None:
        print(f"{n:<26}{'검출실패':>9}")
        continue
    print(f"{n:<26}{e:>9.1f}{s:>9.1f}{sc:>8.3f}{fw:>8.0f}{cos[0]:>9.4f}{cos[1]:>9.4f}")

In [ ]:
N_ORDER = [p.stem for p in n_files if p.stem != "normal_10_salon_wave"]

fig, axes = plt.subplots(2, 9, figsize=(30, 8))
for c, name in enumerate(N_ORDER):
    srcs = (
        (N_DIRS["affine"] / f"c3_{name}.png", "현재"),
        (N_DIRS["none"] / f"c3_{name}.png", "정렬 없음"),
    )
    for r, (p, lab) in enumerate(srcs):
        axes[r, c].imshow(face_crop(Image.open(p).convert("RGB"), pad=1.2, size=340))
        axes[r, c].set_title(f"{lab}  {name[7:17]}", fontsize=8)
        axes[r, c].axis("off")

plt.tight_layout()
savefig(102, "normal_align_compare")
plt.show()

## 정렬 유무 × 복원 비교

정렬을 빼면 입력 얼굴이 덜 망가지므로 복원이 덜 필요할 수 있다. w 0.3 을 고른
근거가 "정렬이 망가뜨린 얼굴을 되돌리려면 그만큼 필요했다" 였으므로, 정렬 제거
후에는 w 가 달라질 수 있다.

upper 6장에 대해 정렬 있음·없음 각각을 w 0.3·0.7 로 복원해 네 조건을 비교한다.
0.5 는 앞선 스윕에서 0.3 과 0.7 사이에 놓이는 것이 확인돼 양 끝만 본다.

In [ ]:
CF_ROOT = Path("/content/cf_align")
%cd /content/CodeFormer

for tag in ("affine", "none"):
    for w in (0.3, 0.7):
        !python inference_codeformer.py -w {w} -i {ALIGN_DIRS[tag]} -o {CF_ROOT}/{tag}_w{w} 2>&1 | tail -1

%cd /content/SalonCutAI/backend
!find {CF_ROOT} -name "*.png" -path "*final_results*" | wc -l

In [ ]:
LABS4 = ("정렬O 복원전", "정렬O w0.3", "정렬X 복원전", "정렬X w0.3")

fig, axes = plt.subplots(4, 6, figsize=(24, 18))
for c, name in enumerate(ORDER):
    srcs = (
        ALIGN_DIRS["affine"] / f"c3_{name}.png",
        CF_ROOT / "affine_w0.3" / "final_results" / f"c3_{name}.png",
        ALIGN_DIRS["none"] / f"c3_{name}.png",
        CF_ROOT / "none_w0.3" / "final_results" / f"c3_{name}.png",
    )
    for r, (lab, p) in enumerate(zip(LABS4, srcs)):
        axes[r, c].imshow(face_crop(Image.open(p).convert("RGB"), pad=1.2, size=400))
        axes[r, c].set_title(f"{lab}  {name[6:16]}", fontsize=9)
        axes[r, c].axis("off")

plt.tight_layout()
savefig(103, "align_x_restore_compare")
plt.show()

In [ ]:
LABS_W = ("정렬X 복원전", "정렬X w0.3", "정렬X w0.7")

fig, axes = plt.subplots(3, 6, figsize=(24, 14))
for c, name in enumerate(ORDER):
    srcs = (
        ALIGN_DIRS["none"] / f"c3_{name}.png",
        CF_ROOT / "none_w0.3" / "final_results" / f"c3_{name}.png",
        CF_ROOT / "none_w0.7" / "final_results" / f"c3_{name}.png",
    )
    for r, (lab, p) in enumerate(zip(LABS_W, srcs)):
        axes[r, c].imshow(face_crop(Image.open(p).convert("RGB"), pad=1.2, size=400))
        axes[r, c].set_title(f"{lab}  {name[6:16]}", fontsize=9)
        axes[r, c].axis("off")

plt.tight_layout()
savefig(104, "noalign_w_compare")
plt.show()

## 환경 재구축

런타임이 끊긴 뒤 이 셀부터 실행한다. Drive 마운트, 한글 폰트, CodeFormer 설치까지
한 번에 처리한다.

CodeFormer 는 Python 3.13 에서 basicsr/setup.py 의 exec 스코프 문제로 설치가
실패한다. 그 부분을 패치한 뒤 설치한다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!apt-get install -qq fonts-nanum > /dev/null
!fc-cache -fv > /dev/null

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

%cd /content
!git clone -q https://github.com/sczhou/CodeFormer.git
%cd /content/CodeFormer
!pip install -q -r requirements.txt 2>&1 | tail -2

import pathlib

p = pathlib.Path("/content/CodeFormer/basicsr/setup.py")
s = p.read_text()
old = "        exec(compile(f.read(), version_file, 'exec'))\n    return locals()['__version__']"
new = "        ns = {}\n        exec(compile(f.read(), version_file, 'exec'), ns)\n    return ns['__version__']"
p.write_text(s.replace(old, new))

!python basicsr/setup.py develop 2>&1 | tail -2
!python scripts/download_pretrained_models.py facelib 2>&1 | tail -1
!python scripts/download_pretrained_models.py CodeFormer 2>&1 | tail -1

print("준비 완료")

In [ ]:
from pathlib import Path

import numpy as np
from PIL import Image

BASE = Path("/content/drive/MyDrive/saloncut_data")
UPPER = BASE / "test_images/upper"
FIG = BASE / "outputs/report_figures"

ALIGN_DIRS = {
    tag: BASE / f"outputs/upper_combo3/align_{tag}"
    for tag in ("affine", "shift", "none")
}
CF_ROOT = Path("/content/cf_align")

ORDER = [
    "upper_06_asian_male", "upper_05_asian_glasses", "upper_04_asian_landscape",
    "upper_02_west_small_face", "upper_03_asian_tied", "upper_01_asian_long_dark",
]


def savefig(num, name):
    p = FIG / f"fig{num}_{name}.png"
    plt.savefig(p, dpi=120, bbox_inches="tight")
    print(f"저장  {p.name}")


!ls {ALIGN_DIRS["none"]}

In [ ]:
%cd /content/CodeFormer
for tag in ("affine", "none"):
    for w in (0.3, 0.7):
        !python inference_codeformer.py -w {w} -i {ALIGN_DIRS[tag]} -o {CF_ROOT}/{tag}_w{w} 2>&1 | tail -1

!find {CF_ROOT} -path "*final_results*" -name "*.png" | wc -l

In [ ]:
!pip install -q insightface onnxruntime-gpu 2>&1 | tail -2

import insightface

app = insightface.app.FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(640, 640))


def face_crop(img, pad=1.2, size=400):
    det = app.get(np.array(img)[:, :, ::-1])
    if not det:
        return None
    x1, y1, x2, y2 = det[0].bbox
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    half = max(x2 - x1, y2 - y1) / 2 * pad
    box = (
        max(0, int(cx - half)), max(0, int(cy - half)),
        min(img.width, int(cx + half)), min(img.height, int(cy + half)),
    )
    c = img.crop(box)
    return c.resize((size, int(size * c.height / c.width)), Image.LANCZOS)


print("준비 완료")

In [ ]:
import shutil

CF_DRIVE = Path("/content/drive/MyDrive/saloncut_data/outputs/upper_restore/align_x_restore")
for tag in ("affine", "none"):
    for w in (0.3, 0.7):
        dst = CF_DRIVE / f"{tag}_w{w}"
        dst.mkdir(parents=True, exist_ok=True)
        for p in (CF_ROOT / f"{tag}_w{w}" / "final_results").glob("*.png"):
            shutil.copy(p, dst / p.name)

print(f"저장  {CF_DRIVE}")

In [ ]:
import cv2


def lap_var(img, det_app):
    """얼굴 영역 라플라시안 분산. 질감 지표."""
    d = det_app.get(np.array(img)[:, :, ::-1])
    if not d:
        return None
    x1, y1, x2, y2 = [int(v) for v in d[0].bbox]
    g = cv2.cvtColor(np.array(img.crop((x1, y1, x2, y2))), cv2.COLOR_RGB2GRAY)
    return float(cv2.Laplacian(g, cv2.CV_64F).var())


def cos_sim(a, b, det_app):
    e = []
    for im in (a, b):
        f = det_app.get(np.array(im)[:, :, ::-1])
        if not f:
            return None
        e.append(f[0].normed_embedding)
    return float(np.dot(e[0], e[1]))


REF = BASE / "ref_faces"
REF_MAP = {"upper_06_asian_male": "ref-05"}

print(f"{'file':<26}{'질감전':>8}{'w0.3':>8}{'w0.7':>8}{'참조0.3':>9}{'참조0.7':>9}")
for name in ORDER:
    ref = Image.open(REF / f"{REF_MAP.get(name, 'ref-01')}.png").convert("RGB")
    pre = Image.open(ALIGN_DIRS["none"] / f"c3_{name}.png").convert("RGB")
    w3 = Image.open(CF_ROOT / "none_w0.3" / "final_results" / f"c3_{name}.png").convert("RGB")
    w7 = Image.open(CF_ROOT / "none_w0.7" / "final_results" / f"c3_{name}.png").convert("RGB")

    print(
        f"{name:<26}{lap_var(pre, app):>8.1f}{lap_var(w3, app):>8.1f}"
        f"{lap_var(w7, app):>8.1f}"
        f"{cos_sim(ref, w3, app):>9.4f}{cos_sim(ref, w7, app):>9.4f}"
    )

In [ ]:
%cd /content
!git clone -q https://github.com/qja0707/SalonCutAI.git
%cd /content/SalonCutAI/backend
!pip install -q diffusers transformers accelerate mediapipe 2>&1 | tail -2

import os
import sys

os.environ["SALON_STORAGE_DIR"] = "/content/storage"
os.environ["IMAGE_GEN_ENABLED"] = "1"
sys.path.insert(0, "/content/SalonCutAI/backend")

from src.ai_engine.image_gen import combo3, compose, downloads, loader as gen_loader, masks

downloads.ensure_models()
print("준비 완료")

In [ ]:
import cv2

SEED = 42
DILATE_RATIO = 0.077
name = "upper_02_west_small_face"

REF23 = BASE / "ref_faces/ref-23.png"
OUT23 = BASE / "outputs/upper_combo3/align_none_ref23"
OUT23.mkdir(parents=True, exist_ok=True)

out, img_r, _ = combo3.generate(
    Image.open(UPPER / f"{name}.jpg").convert("RGB"), REF23, SEED
)

det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
fw = det[0].bbox[2] - det[0].bbox[0]

face_mask = masks.build_face_mask(img_r)
hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
gen_mask = masks.build_gen_mask(face_mask, hair_mask)

b = compose.color_transfer(out, img_r, gen_mask)
c = compose.recompose_with_hair(img_r, b.resize(img_r.size), face_mask, hair_mask)
d = compose.transfer_high_freq(c, img_r, gen_mask)

d.save(OUT23 / f"c3_{name}.png")
out.save(OUT23 / f"gen_{name}.png")
img_r.save(OUT23 / f"src_{name}.png")
print("완료")

In [ ]:
CF23 = Path("/content/cf_ref23")

%cd /content/CodeFormer
for w in (0.3, 0.7):
    !python inference_codeformer.py -w {w} -i {OUT23} -o {CF23}/w{w} 2>&1 | tail -1
%cd /content/SalonCutAI/backend

LABS = (
    ("ref-01 복원전", ALIGN_DIRS["none"] / f"c3_{name}.png"),
    ("ref-01 w0.3", CF_ROOT / "none_w0.3" / "final_results" / f"c3_{name}.png"),
    ("ref-01 w0.7", CF_ROOT / "none_w0.7" / "final_results" / f"c3_{name}.png"),
    ("ref-23 복원전", OUT23 / f"c3_{name}.png"),
    ("ref-23 w0.3", CF23 / "w0.3" / "final_results" / f"c3_{name}.png"),
    ("ref-23 w0.7", CF23 / "w0.7" / "final_results" / f"c3_{name}.png"),
)

fig, axes = plt.subplots(1, 6, figsize=(24, 5))
for ax, (lab, p) in zip(axes, LABS):
    ax.imshow(face_crop(Image.open(p).convert("RGB"), pad=1.2, size=400))
    ax.set_title(lab, fontsize=10)
    ax.axis("off")
plt.tight_layout()
savefig(105, "upper02_ref23_restore")
plt.show()

## 실사용 사진 생성

혜리님이 공유한 실제 매장 사진 5장이다. 비율 0.172~0.212, latent 4.2~5.0 으로
실사용 구간이 좁게 확정됐다. 5장 중 4장이 얼굴을 비스듬히 하고 시선이 옆인데,
머리 실루엣을 보여주는 촬영 구도라 구조적인 특성으로 보인다.

정렬 제거 + 헤어 팽창 비율 0.077 로 생성한다. 참조는 전원 여성이므로 ref-01 로
고정한다.

초상권 사진이므로 수치와 내부 판단에만 쓰고 보고서·디스커션에는 수록하지 않는다.

In [ ]:
SALON = BASE / "test_images/salon"
SALON_OUT = BASE / "outputs/salon_none"
SALON_OUT.mkdir(parents=True, exist_ok=True)

import time

s_files = sorted(SALON.glob("*.jpg"))
print(f"{len(s_files)}장")

for path in s_files:
    t0 = time.time()
    out, img_r, _ = combo3.generate(
        Image.open(path).convert("RGB"), REF / "ref-01.png", SEED
    )

    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]

    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    b = compose.color_transfer(out, img_r, gen_mask)
    c = compose.recompose_with_hair(img_r, b.resize(img_r.size), face_mask, hair_mask)
    d = compose.transfer_high_freq(c, img_r, gen_mask)

    d.save(SALON_OUT / f"c3_{path.stem}.png")
    out.save(SALON_OUT / f"gen_{path.stem}.png")
    img_r.save(SALON_OUT / f"src_{path.stem}.png")
    print(f"{path.stem:<30} 얼굴폭 {int(fw):>3}  {time.time() - t0:5.1f}초")

print("\n생성 완료")

In [ ]:
import gc

import torch

from src.ai_engine.image_gen import loader as gen_loader

gen_loader._combo3 = None
gen_loader._combo5 = None
gc.collect()
torch.cuda.empty_cache()

print(f"할당 {torch.cuda.memory_allocated() / 1e9:.2f}GB  "
      f"예약 {torch.cuda.memory_reserved() / 1e9:.2f}GB")

In [ ]:
CF_SALON = Path("/content/cf_salon")

%cd /content/CodeFormer
for w in (0.3, 0.7):
    !python inference_codeformer.py -w {w} -i {SALON_OUT} -o {CF_SALON}/w{w} 2>&1 | tail -2
%cd /content/SalonCutAI/backend

!find {CF_SALON} -path "*final_results*" -name "c3_*.png" | wc -l

In [ ]:
S_ORDER = [p.stem for p in s_files]
LABS_S = ("1024 원본", "생성", "복원전 최종", "w0.3", "w0.7")

fig, axes = plt.subplots(5, 5, figsize=(20, 19))
for c, name in enumerate(S_ORDER):
    srcs = (
        SALON_OUT / f"src_{name}.png",
        SALON_OUT / f"gen_{name}.png",
        SALON_OUT / f"c3_{name}.png",
        CF_SALON / "w0.3" / "final_results" / f"c3_{name}.png",
        CF_SALON / "w0.7" / "final_results" / f"c3_{name}.png",
    )
    for r, (lab, p) in enumerate(zip(LABS_S, srcs)):
        cr = face_crop(Image.open(p).convert("RGB"), pad=1.2, size=380)
        axes[r, c].imshow(cr if cr else Image.new("RGB", (380, 380)))
        axes[r, c].set_title(f"{lab}  {name[6:16]}", fontsize=8)
        axes[r, c].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import time

SALON_AFF = BASE / "outputs/salon_affine"
SALON_AFF.mkdir(parents=True, exist_ok=True)

for path in s_files:
    t0 = time.time()
    out, img_r, _ = combo3.generate(
        Image.open(path).convert("RGB"), REF / "ref-01.png", SEED
    )

    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]

    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    b = compose.color_transfer(out, img_r, gen_mask)
    c = compose.align_then_recompose(img_r, b, face_mask, hair_mask)   # 정렬 있음
    d = compose.transfer_high_freq(c, img_r, gen_mask)

    d.save(SALON_AFF / f"c3_{path.stem}.png")
    print(f"{path.stem:<30} {time.time() - t0:5.1f}초")

print("\n생성 완료")

In [ ]:
import gc

import torch

gen_loader._combo3 = None
gen_loader._combo5 = None
gc.collect()
torch.cuda.empty_cache()

CF_AFF = Path("/content/cf_salon_aff")
%cd /content/CodeFormer
!python inference_codeformer.py -w 0.7 -i {SALON_AFF} -o {CF_AFF} 2>&1 | tail -2
%cd /content/SalonCutAI/backend

In [ ]:
LABS_AB = ("원본", "정렬O + 복원", "정렬X + 복원")

# 전신
fig, axes = plt.subplots(3, 5, figsize=(20, 22))
for c, name in enumerate(S_ORDER):
    srcs = (
        SALON_OUT / f"src_{name}.png",
        CF_AFF / "final_results" / f"c3_{name}.png",
        CF_SALON / "w0.7" / "final_results" / f"c3_{name}.png",
    )
    for r, (lab, p) in enumerate(zip(LABS_AB, srcs)):
        axes[r, c].imshow(Image.open(p))
        axes[r, c].set_title(f"{lab}  {name[6:16]}", fontsize=9)
        axes[r, c].axis("off")
plt.tight_layout()
plt.show()

# 얼굴 확대
fig, axes = plt.subplots(3, 5, figsize=(20, 13))
for c, name in enumerate(S_ORDER):
    srcs = (
        SALON_OUT / f"src_{name}.png",
        CF_AFF / "final_results" / f"c3_{name}.png",
        CF_SALON / "w0.7" / "final_results" / f"c3_{name}.png",
    )
    for r, (lab, p) in enumerate(zip(LABS_AB, srcs)):
        cr = face_crop(Image.open(p).convert("RGB"), pad=1.2, size=380)
        axes[r, c].imshow(cr if cr else Image.new("RGB", (380, 380)))
        axes[r, c].set_title(f"{lab}  {name[6:16]}", fontsize=9)
        axes[r, c].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
SALON_B2 = BASE / "outputs/salon_none_b2"
SALON_B2.mkdir(parents=True, exist_ok=True)

for path in s_files:
    name = path.stem
    img_r = Image.open(SALON_OUT / f"src_{name}.png").convert("RGB")
    rest = Image.open(
        CF_SALON / "w0.7" / "final_results" / f"c3_{name}.png"
    ).convert("RGB").resize(img_r.size)

    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]
    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    shifted = compose.color_transfer(rest, img_r, gen_mask)
    b2 = Image.composite(shifted, rest, gen_mask.convert("L"))
    b2.save(SALON_B2 / f"c3_{name}.png")

    m = np.array(gen_mask) > 127
    de = delta_e(rest, b2)
    print(f"{name:<30} 마스크안 {de[m].mean():>5.2f}  밖 {de[~m].mean():>5.2f}")

In [ ]:
LABS_AB = ("원본", "정렬O + 복원", "정렬X + 복원", "정렬X + 복원 + 색정합")


def grid(crop, figsize):
    fig, axes = plt.subplots(4, 5, figsize=figsize)
    for c, name in enumerate(S_ORDER):
        srcs = (
            SALON_OUT / f"src_{name}.png",
            CF_AFF / "final_results" / f"c3_{name}.png",
            CF_SALON / "w0.7" / "final_results" / f"c3_{name}.png",
            SALON_B2 / f"c3_{name}.png",
        )
        for r, (lab, p) in enumerate(zip(LABS_AB, srcs)):
            im = Image.open(p).convert("RGB")
            if crop:
                cr = face_crop(im, pad=1.2, size=380)
                im = cr if cr else Image.new("RGB", (380, 380))
            axes[r, c].imshow(im)
            axes[r, c].set_title(f"{lab}  {name[6:16]}", fontsize=8)
            axes[r, c].axis("off")
    plt.tight_layout()
    plt.show()


grid(False, (20, 29))   # 전신
grid(True, (20, 17))    # 얼굴 확대

In [ ]:
import glob

paths = sorted(glob.glob(f"{REF}/ref-*.png"))
fig, axes = plt.subplots(4, 8, figsize=(20, 11))
for ax, p in zip(axes.ravel(), paths):
    ax.imshow(Image.open(p))
    ax.set_title(p.split("/")[-1].replace(".png", ""), fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import time

REF_LIST = ("ref-02", "ref-09")
REF_DIRS = {}

for r in REF_LIST:
    d = BASE / f"outputs/salon_{r}"
    d.mkdir(parents=True, exist_ok=True)
    REF_DIRS[r] = d

for r in REF_LIST:
    for path in s_files:
        out, img_r, _ = combo3.generate(
            Image.open(path).convert("RGB"), REF / f"{r}.png", SEED
        )

        det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
        fw = det[0].bbox[2] - det[0].bbox[0]

        face_mask = masks.build_face_mask(img_r)
        hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
        gen_mask = masks.build_gen_mask(face_mask, hair_mask)

        b = compose.color_transfer(out, img_r, gen_mask)
        c = compose.recompose_with_hair(img_r, b.resize(img_r.size), face_mask, hair_mask)
        d_img = compose.transfer_high_freq(c, img_r, gen_mask)

        d_img.save(REF_DIRS[r] / f"c3_{path.stem}.png")
    print(f"{r} 완료")

In [ ]:
import gc

import torch

gen_loader._combo3 = None
gen_loader._combo5 = None
gc.collect()
torch.cuda.empty_cache()

CF_REF = {}
%cd /content/CodeFormer
for r in REF_LIST:
    CF_REF[r] = Path(f"/content/cf_{r}")
    !python inference_codeformer.py -w 0.7 -i {REF_DIRS[r]} -o {CF_REF[r]} 2>&1 | tail -1
%cd /content/SalonCutAI/backend

# B2 색 정합
B2_REF = {}
for r in REF_LIST:
    B2_REF[r] = BASE / f"outputs/salon_{r}_b2"
    B2_REF[r].mkdir(parents=True, exist_ok=True)

    for path in s_files:
        name = path.stem
        img_r = Image.open(SALON_OUT / f"src_{name}.png").convert("RGB")
        rest = Image.open(
            CF_REF[r] / "final_results" / f"c3_{name}.png"
        ).convert("RGB").resize(img_r.size)

        det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
        fw = det[0].bbox[2] - det[0].bbox[0]
        face_mask = masks.build_face_mask(img_r)
        hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
        gen_mask = masks.build_gen_mask(face_mask, hair_mask)

        shifted = compose.color_transfer(rest, img_r, gen_mask)
        Image.composite(shifted, rest, gen_mask.convert("L")).save(
            B2_REF[r] / f"c3_{name}.png"
        )
    print(f"{r} B2 완료")

In [ ]:
ROWS = (
    ("원본", None),
    ("ref-01", SALON_B2),
    ("ref-02", B2_REF["ref-02"]),
    ("ref-09", B2_REF["ref-09"]),
)

fig, axes = plt.subplots(4, 6, figsize=(24, 17))
for r, (lab, d) in enumerate(ROWS):
    # 첫 열은 참조 얼굴
    if d is None:
        axes[r, 0].axis("off")
    else:
        rid = lab
        axes[r, 0].imshow(face_crop(Image.open(REF / f"{rid}.png").convert("RGB"),
                                    pad=1.3, size=380))
        axes[r, 0].set_title(f"참조 {rid}", fontsize=9)
    axes[r, 0].axis("off")

    for c, name in enumerate(S_ORDER, start=1):
        p = SALON_OUT / f"src_{name}.png" if d is None else d / f"c3_{name}.png"
        cr = face_crop(Image.open(p).convert("RGB"), pad=1.2, size=380)
        axes[r, c].imshow(cr if cr else Image.new("RGB", (380, 380)))
        axes[r, c].set_title(f"{lab}  {name[6:16]}", fontsize=8)
        axes[r, c].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
name = "salon_01_long_wave_brown"   # 가장 뚜렷한 걸로 바꿔도 됨

EYE_STAGES = (
    (SALON_OUT / f"src_{name}.png", "1024 원본"),
    (SALON_OUT / f"gen_{name}.png", "생성"),
    (SALON_OUT / f"c3_{name}.png", "후처리 완료"),
    (CF_SALON / "w0.7" / "final_results" / f"c3_{name}.png", "복원"),
    (SALON_B2 / f"c3_{name}.png", "복원+색정합"),
)


def eye_crop(img, size=520):
    det = app.get(np.array(img)[:, :, ::-1])
    if not det:
        return None
    kps = det[0].kps
    xs, ys = kps[:2, 0], kps[:2, 1]
    cx, cy = xs.mean(), ys.mean()
    half_w = abs(xs[1] - xs[0]) * 1.1
    half_h = half_w * 0.42
    box = (
        max(0, int(cx - half_w)), max(0, int(cy - half_h)),
        min(img.width, int(cx + half_w)), min(img.height, int(cy + half_h)),
    )
    c = img.crop(box)
    return c.resize((size, int(size * c.height / c.width)), Image.LANCZOS)


fig, axes = plt.subplots(1, 5, figsize=(24, 5))
for ax, (p, lab) in zip(axes, EYE_STAGES):
    cr = eye_crop(Image.open(p).convert("RGB"))
    ax.imshow(cr if cr else Image.new("RGB", (520, 220)))
    ax.set_title(lab, fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
PAIRS = ("salon_03_long_wave_ring", "salon_04_long_wave_black")
STAGES4 = (
    (SALON_OUT, "src_{}.png", "1024 원본"),
    (SALON_OUT, "gen_{}.png", "생성"),
    (SALON_OUT, "c3_{}.png", "후처리(복원 전)"),
    (SALON_B2, "c3_{}.png", "복원 후"),
)

fig, axes = plt.subplots(2, 4, figsize=(22, 9))
for r, name in enumerate(PAIRS):
    for c, (d, pat, lab) in enumerate(STAGES4):
        cr = eye_crop(Image.open(d / pat.format(name)).convert("RGB"))
        axes[r, c].imshow(cr if cr else Image.new("RGB", (520, 220)))
        axes[r, c].set_title(f"{lab}  {name[6:14]}", fontsize=10)
        axes[r, c].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
NOHF = BASE / "outputs/salon_nohf"
NOHF.mkdir(parents=True, exist_ok=True)

for path in s_files:
    name = path.stem
    img_r = Image.open(SALON_OUT / f"src_{name}.png").convert("RGB")
    gen = Image.open(SALON_OUT / f"gen_{name}.png").convert("RGB")

    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]
    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    b = compose.color_transfer(gen, img_r, gen_mask)
    c = compose.recompose_with_hair(img_r, b.resize(img_r.size), face_mask, hair_mask)
    c.save(NOHF / f"c3_{name}.png")     # 고주파 생략
    print(f"{name} 완료")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for r, name in enumerate(PAIRS):
    srcs = (
        (SALON_OUT / f"gen_{name}.png", "생성"),
        (SALON_OUT / f"c3_{name}.png", "고주파 0.5"),
        (NOHF / f"c3_{name}.png", "고주파 없음"),
    )
    for c, (p, lab) in enumerate(srcs):
        cr = eye_crop(Image.open(p).convert("RGB"))
        axes[r, c].imshow(cr if cr else Image.new("RGB", (520, 220)))
        axes[r, c].set_title(f"{lab}  {name[6:14]}", fontsize=10)
        axes[r, c].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
NOHF_CF = Path("/content/cf_nohf")

%cd /content/CodeFormer
!python inference_codeformer.py -w 0.7 -i {NOHF} -o {NOHF_CF} 2>&1 | tail -1
%cd /content/SalonCutAI/backend

fig, axes = plt.subplots(5, 4, figsize=(22, 22))
COLS = (
    (SALON_OUT, "src_{}.png", "원본"),
    (SALON_OUT, "gen_{}.png", "생성"),
    (SALON_B2, "c3_{}.png", "고주파 0.5 + 복원"),
    (NOHF_CF / "final_results", "c3_{}.png", "고주파 없음 + 복원"),
)

for r, name in enumerate(S_ORDER):
    for c, (d, pat, lab) in enumerate(COLS):
        cr = eye_crop(Image.open(d / pat.format(name)).convert("RGB"))
        axes[r, c].imshow(cr if cr else Image.new("RGB", (520, 220)))
        axes[r, c].set_title(f"{lab}  {name[6:14]}", fontsize=9)
        axes[r, c].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(18, 22))
COLS3 = (
    (SALON_OUT, "src_{}.png", "원본"),
    (SALON_B2, "c3_{}.png", "ref-01"),
    (B2_REF["ref-09"], "c3_{}.png", "ref-09"),
)
for r, name in enumerate(S_ORDER):
    for c, (d, pat, lab) in enumerate(COLS3):
        cr = eye_crop(Image.open(d / pat.format(name)).convert("RGB"))
        axes[r, c].imshow(cr if cr else Image.new("RGB", (520, 220)))
        axes[r, c].set_title(f"{lab}  {name[6:14]}", fontsize=9)
        axes[r, c].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
REF09_NOHF = BASE / "outputs/salon_ref09_nohf"
REF09_NOHF.mkdir(parents=True, exist_ok=True)

for path in s_files:
    out, img_r, _ = combo3.generate(
        Image.open(path).convert("RGB"), REF / "ref-09.png", SEED
    )

    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]
    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    b = compose.color_transfer(out, img_r, gen_mask)
    c = compose.recompose_with_hair(img_r, b.resize(img_r.size), face_mask, hair_mask)
    c.save(REF09_NOHF / f"c3_{path.stem}.png")     # 고주파 생략
    print(f"{path.stem} 완료")

In [ ]:
import gc

import torch

gen_loader._combo3 = None
gen_loader._combo5 = None
gc.collect()
torch.cuda.empty_cache()

R9_CF = Path("/content/cf_ref09_nohf")
%cd /content/CodeFormer
!python inference_codeformer.py -w 0.7 -i {REF09_NOHF} -o {R9_CF} 2>&1 | tail -1
%cd /content/SalonCutAI/backend

R9_B2 = BASE / "outputs/salon_ref09_nohf_b2"
R9_B2.mkdir(parents=True, exist_ok=True)

for path in s_files:
    name = path.stem
    img_r = Image.open(SALON_OUT / f"src_{name}.png").convert("RGB")
    rest = Image.open(R9_CF / "final_results" / f"c3_{name}.png").convert("RGB").resize(img_r.size)

    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]
    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    shifted = compose.color_transfer(rest, img_r, gen_mask)
    Image.composite(shifted, rest, gen_mask.convert("L")).save(R9_B2 / f"c3_{name}.png")

print("완료")

In [ ]:
COLS4 = (
    (SALON_OUT, "src_{}.png", "원본"),
    (B2_REF["ref-09"], "c3_{}.png", "ref-09 고주파O"),
    (R9_B2, "c3_{}.png", "ref-09 고주파X"),
)

fig, axes = plt.subplots(3, 5, figsize=(20, 22))
for c, name in enumerate(S_ORDER):
    for r, (d, pat, lab) in enumerate(COLS4):
        axes[r, c].imshow(Image.open(d / pat.format(name)))
        axes[r, c].set_title(f"{lab}  {name[6:16]}", fontsize=9)
        axes[r, c].axis("off")
plt.tight_layout()
plt.show()

## 복원 모델 비교 — 설치

GFPGAN v1.4 와 RestoreFormer 를 CodeFormer 와 같은 조건으로 돌린다. 둘 다
gfpgan 패키지의 GFPGANer 로 로드되고, facexlib 로 얼굴을 검출·정렬해 512 로
복원한 뒤 되붙이는 구조가 CodeFormer 와 같아 조건이 맞는다.

basicsr 는 CodeFormer 의 develop 설치본(1.3.2)을 그대로 쓴다. pip 로 다시 깔면
CodeFormer 가 깨지므로 gfpgan 은 --no-deps 로 넣는다.

빠져 있던 basicsr.ops 는 CUDA 확장 import 실패를 try/except 로 삼키게 짜여 있어
파이썬 파일만 채우면 import 가 통과한다. GFPGAN v1.4 는 arch='clean' 이라
순수 PyTorch 경로(stylegan2_clean_arch)만 타고 FusedLeakyReLU 를 만들지 않는다.
컴파일이 필요 없는 이유다.

In [ ]:
import urllib.request

BSR = Path("/content/CodeFormer/basicsr")
RAW = "https://raw.githubusercontent.com/XPixelGroup/BasicSR/master/basicsr"

OPS = [
    "ops/__init__.py",
    "ops/fused_act/__init__.py",
    "ops/fused_act/fused_act.py",
    "ops/upfirdn2d/__init__.py",
    "ops/upfirdn2d/upfirdn2d.py",
    "ops/dcn/__init__.py",
    "ops/dcn/deform_conv.py",
]

for rel in OPS:
    dst = BSR / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f"{RAW}/{rel}", dst)
    print(f"{rel:<32}{dst.stat().st_size:>7} bytes")

!pip install -q --no-deps gfpgan 2>&1 | tail -3

import basicsr
from gfpgan import GFPGANer

print("\nbasicsr  ", basicsr.__file__)
print("gfpgan import 성공")

In [ ]:
# CodeFormer 와 gfpgan 이 같은 이름의 arch 를 각자 등록한다.
# 추론은 클래스를 직접 만들어 쓰므로 registry 충돌은 무해하다. 먼저 등록된
# CodeFormer 쪽을 남기고 중복만 건너뛴다. 무엇이 겹쳤는지는 찍어서 남긴다.
import basicsr
from basicsr.utils.registry import ARCH_REGISTRY

if not hasattr(ARCH_REGISTRY, "_skip_dup"):
    _orig = ARCH_REGISTRY._do_register

    def _skip_dup(name, obj):
        if name in ARCH_REGISTRY._obj_map:
            print(f"  중복 건너뜀  {name}")
            return
        _orig(name, obj)

    ARCH_REGISTRY._do_register = _skip_dup
    ARCH_REGISTRY._skip_dup = True

from gfpgan import GFPGANer

print("\nbasicsr  ", basicsr.__file__)
print("gfpgan   ", GFPGANer.__module__)
print("gfpgan import 성공")

In [ ]:
import importlib.util

from basicsr.utils.registry import ARCH_REGISTRY

# gfpgan 의 data·models 는 학습 전용이고 CodeFormer 의 basicsr 1.3.2 에는
# 학습 쪽 모듈이 없다. utils.py 가 둘을 참조하지 않으므로 스캔만 막는다.
GF = Path(importlib.util.find_spec("gfpgan").submodule_search_locations[0])
for sub in ("data", "models"):
    (GF / sub / "__init__.py").write_text("")
    print(f"비움  gfpgan/{sub}/__init__.py")

# CodeFormer 와 gfpgan 이 같은 이름의 arch 를 각자 등록한다. 추론은 클래스를
# 직접 만들어 쓰므로 충돌은 무해하다. 먼저 등록된 쪽을 남기고 중복만 건너뛴다.
if not hasattr(ARCH_REGISTRY, "_skip_dup"):
    _orig = ARCH_REGISTRY._do_register

    def _skip_dup(name, obj):
        if name in ARCH_REGISTRY._obj_map:
            print(f"  중복 건너뜀  {name}")
            return
        _orig(name, obj)

    ARCH_REGISTRY._do_register = _skip_dup
    ARCH_REGISTRY._skip_dup = True

import basicsr
from gfpgan import GFPGANer

print("\nbasicsr  ", basicsr.__file__)
print("GFPGANer ", GFPGANer)

## 복원 모델 비교 — 실행

CodeFormer w0.7 과 GFPGAN v1.4 · RestoreFormer 를 같은 5장에 적용한다.
입력은 고주파 이식을 뺀 salon 결과(NOHF)로, 확정 파이프라인의 복원 직전 상태다.

GFPGANer 는 arch 만 바꿔 두 모델을 같은 코드로 로드한다. 인자는 CodeFormer
inference 기본값과 같다 — upscale 2, retinaface_resnet50 검출, only_center_face
False, 얼굴 512 정렬 후 되붙이기.

In [ ]:
import gc
import time

import cv2
import torch

WEIGHTS = Path("/content/restore_weights")
WEIGHTS.mkdir(exist_ok=True)

MODELS = {
    "gfpgan": (
        "GFPGANv1.4.pth",
        "clean",
        "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",
    ),
    "restoreformer": (
        "RestoreFormer.pth",
        "RestoreFormer",
        "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/RestoreFormer.pth",
    ),
}

for tag, (fname, _, url) in MODELS.items():
    dst = WEIGHTS / fname
    if not dst.exists():
        urllib.request.urlretrieve(url, dst)
    print(f"{tag:<16}{dst.stat().st_size / 1e6:>8.1f} MB")

# SDXL 이 올라와 있으면 복원 모델과 VRAM 이 충돌한다
gen_loader._combo3 = None
gen_loader._combo5 = None
gc.collect()
torch.cuda.empty_cache()

RESTORE = BASE / "outputs/salon_restore"

for tag, (fname, arch, _) in MODELS.items():
    out_dir = RESTORE / tag
    out_dir.mkdir(parents=True, exist_ok=True)

    restorer = GFPGANer(
        model_path=str(WEIGHTS / fname), upscale=2, arch=arch, bg_upsampler=None
    )

    print()
    for path in sorted(NOHF.glob("*.png")):
        t = time.time()
        _, _, restored = restorer.enhance(
            cv2.imread(str(path)),
            has_aligned=False,
            only_center_face=False,
            paste_back=True,
        )
        cv2.imwrite(str(out_dir / path.name), restored)
        print(f"{tag:<16}{path.stem:<28}{time.time() - t:>6.1f}s")

    del restorer
    gc.collect()
    torch.cuda.empty_cache()

## 복원 모델 비교 — 지표와 눈 확대

복원 없음 · CodeFormer w0.7 · GFPGAN v1.4 · RestoreFormer 네 가지를 나란히 본다.

세 지표를 쓴다. 질감(라플라시안 분산)은 "로봇 같음" 지적과 직결되고, 참조
코사인은 참조 얼굴을 얼마나 따라갔는지, 원본 코사인은 초상권 회피가 유지되는지를
본다. 복원 출력이 upscale 2 로 2배라 원본 크기로 맞춘 뒤 잰다.

수치는 어느 쪽이 나은지 가리는 데만 쓰고 판정은 눈 확대로 한다. salon 은
초상권 사진이라 그림을 report_figures 에 저장하지 않는다.

In [ ]:
COLS = {
    "복원없음": NOHF,
    "CodeFormer": NOHF_CF / "final_results",
    "GFPGAN": RESTORE / "gfpgan",
    "RestoreFormer": RESTORE / "restoreformer",
}

ref = Image.open(REF / "ref-01.png").convert("RGB")


def load(d, name, size):
    """복원 출력은 upscale 2 라 원본 크기로 맞춰야 지표가 비교된다."""
    return Image.open(d / f"c3_{name}.png").convert("RGB").resize(size)


for title, fn, fmt in (
    ("질감 — 라플라시안 분산 (클수록 선명)", lambda im, src: lap_var(im, app), "15.1f"),
    ("참조 코사인 — ref-01 (클수록 참조를 닮음)", lambda im, src: cos_sim(ref, im, app), "15.4f"),
    ("원본 코사인 (작을수록 회피 성공)", lambda im, src: cos_sim(src, im, app), "15.4f"),
):
    print(f"\n{title}")
    print(f"{'file':<20}" + "".join(f"{k:>15}" for k in COLS))
    for name in S_ORDER:
        src = Image.open(SALON_OUT / f"src_{name}.png").convert("RGB")
        cells = []
        for d in COLS.values():
            v = fn(load(d, name, src.size), src)
            cells.append(f"{v:>{fmt}}" if v is not None else f"{'검출실패':>15}")
        print(f"{name[6:20]:<20}" + "".join(cells))

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(24, 22))

for r, name in enumerate(S_ORDER):
    src = Image.open(SALON_OUT / f"src_{name}.png").convert("RGB")
    cols = [("원본", src)] + [(k, load(d, name, src.size)) for k, d in COLS.items()]
    for c, (lab, im) in enumerate(cols):
        cr = eye_crop(im)
        axes[r, c].imshow(cr if cr else Image.new("RGB", (520, 220)))
        axes[r, c].set_title(f"{lab}  {name[6:14]}", fontsize=9)
        axes[r, c].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
REF02_NOHF = BASE / "outputs/salon_ref02_nohf"
REF02_NOHF.mkdir(parents=True, exist_ok=True)

for path in s_files:
    out, img_r, _ = combo3.generate(
        Image.open(path).convert("RGB"), REF / "ref-02.png", SEED
    )

    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]
    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    b = compose.color_transfer(out, img_r, gen_mask)
    c = compose.recompose_with_hair(img_r, b.resize(img_r.size), face_mask, hair_mask)
    c.save(REF02_NOHF / f"c3_{path.stem}.png")     # 고주파 생략
    print(f"{path.stem} 완료")

In [ ]:
gen_loader._combo3 = None
gen_loader._combo5 = None
gc.collect()
torch.cuda.empty_cache()

NOHF_SETS = {"ref-01": NOHF, "ref-02": REF02_NOHF, "ref-09": REF09_NOHF}

# --- CodeFormer w0.7 : ref-02 만 새로 ---

CF_REF02 = Path("/content/cf_ref02_nohf")
%cd /content/CodeFormer
!python inference_codeformer.py -w 0.7 -i {REF02_NOHF} -o {CF_REF02} 2>&1 | tail -1
%cd /content/SalonCutAI/backend

CF_NOHF = {
    "ref-01": NOHF_CF / "final_results",
    "ref-02": CF_REF02 / "final_results",
    "ref-09": R9_CF / "final_results",
}

# --- GFPGAN v1.4 : 세 참조 모두 ---

GF_ROOT = BASE / "outputs/salon_restore_gfpgan"
GF_NOHF = {}

restorer = GFPGANer(
    model_path=str(WEIGHTS / "GFPGANv1.4.pth"),
    upscale=2,
    arch="clean",
    bg_upsampler=None,
)

for r, src_dir in NOHF_SETS.items():
    out_dir = GF_ROOT / r
    out_dir.mkdir(parents=True, exist_ok=True)
    GF_NOHF[r] = out_dir

    for path in sorted(src_dir.glob("*.png")):
        _, _, restored = restorer.enhance(
            cv2.imread(str(path)),
            has_aligned=False,
            only_center_face=False,
            paste_back=True,
        )
        cv2.imwrite(str(out_dir / path.name), restored)
    print(f"{r:<10} GFPGAN 5장 완료")

del restorer
gc.collect()
torch.cuda.empty_cache()

## 참조별 CodeFormer vs GFPGAN

ref-01·ref-02·ref-09 세 참조로 만든 결과에 두 복원을 적용해 비교한다.
RestoreFormer 는 세 지표 전부 최하라 뺐다.

ref-01 단독 비교에서는 질감 3승 2패, 회피 4승 1패로 CodeFormer 가 앞섰지만
격차가 작았다. 참조를 바꿔도 같은 방향이면 확정하고, 뒤집히면 참조 의존이
있다는 뜻이라 판단 기준을 다시 잡아야 한다.

각 참조마다 자기 ref 이미지로 참조 코사인을 잰다.

In [ ]:
REFS = ("ref-01", "ref-02", "ref-09")
METHODS = {"복원없음": NOHF_SETS, "CodeFormer": CF_NOHF, "GFPGAN": GF_NOHF}


def pick(m, r, name, size):
    d = METHODS[m][r] if m != "복원없음" else NOHF_SETS[r]
    return Image.open(d / f"c3_{name}.png").convert("RGB").resize(size)


for title, fn, fmt in (
    ("질감 — 라플라시안 분산", lambda im, src, rf: lap_var(im, app), "13.1f"),
    ("참조 코사인 — 각 참조 기준", lambda im, src, rf: cos_sim(rf, im, app), "13.4f"),
    ("원본 코사인 — 작을수록 회피", lambda im, src, rf: cos_sim(src, im, app), "13.4f"),
):
    print(f"\n{title}")
    print(f"{'':<16}" + "".join(f"{f'{r} {m}':>13}" for r in REFS for m in METHODS))
    for name in S_ORDER:
        src = Image.open(SALON_OUT / f"src_{name}.png").convert("RGB")
        cells = []
        for r in REFS:
            rf = Image.open(REF / f"{r}.png").convert("RGB")
            for m in METHODS:
                v = fn(pick(m, r, name, src.size), src, rf)
                cells.append(f"{v:>{fmt}}" if v is not None else f"{'실패':>13}")
        print(f"{name[6:16]:<16}" + "".join(cells))

In [ ]:
r = "ref-02"     # ref-09 로 바꿔 한 번 더 돌린다

COLS_R = (
    ("원본", None),
    ("복원없음", NOHF_SETS[r]),
    ("CodeFormer", CF_NOHF[r]),
    ("GFPGAN", GF_NOHF[r]),
)

fig, axes = plt.subplots(5, 4, figsize=(18, 24))
for row, name in enumerate(S_ORDER):
    src = Image.open(SALON_OUT / f"src_{name}.png").convert("RGB")
    for c, (lab, d) in enumerate(COLS_R):
        im = src if d is None else Image.open(d / f"c3_{name}.png").convert("RGB").resize(src.size)
        cr = face_crop(im, pad=1.6, size=460)
        axes[row, c].imshow(cr if cr else Image.new("RGB", (460, 460)))
        axes[row, c].set_title(f"{lab}  {name[6:14]}", fontsize=9)
        axes[row, c].axis("off")

plt.tight_layout()
plt.suptitle(f"{r} — 얼굴 전체", y=1.002, fontsize=13)
plt.show()

In [ ]:
from mediapipe.tasks.python import vision

BROW_CONNS = (
    vision.FaceLandmarksConnections.FACE_LANDMARKS_LEFT_EYEBROW,
    vision.FaceLandmarksConnections.FACE_LANDMARKS_RIGHT_EYEBROW,
)


def build_brow_mask(img, dilate_ratio=0.0):
    """양 눈썹 영역. 연결이 위·아래 두 갈래로 끊겨 있어 볼록껍질로 채운다."""
    w, h = img.size
    res = gen_loader.get_landmarker().detect(masks._to_mp_image(img))
    if not res.face_landmarks:
        return None
    lm = res.face_landmarks[0]

    mask = np.zeros((h, w), np.uint8)
    for conns in BROW_CONNS:
        idx = sorted({c.start for c in conns} | {c.end for c in conns})
        pts = np.array([[int(lm[i].x * w), int(lm[i].y * h)] for i in idx])
        cv2.fillConvexPoly(mask, cv2.convexHull(pts), 255)

    if dilate_ratio > 0:
        det = gen_loader.get_face_app().get(np.array(img)[:, :, ::-1])
        d = max(1, int((det[0].bbox[2] - det[0].bbox[0]) * dilate_ratio))
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (d * 2 + 1, d * 2 + 1))
        mask = cv2.dilate(mask, k, iterations=1)

    return Image.fromarray(mask)


fig, axes = plt.subplots(1, 5, figsize=(24, 6))
for ax, name in zip(axes, S_ORDER):
    src = Image.open(SALON_OUT / f"src_{name}.png").convert("RGB")
    brow = build_brow_mask(src)

    ov = np.array(src).astype(np.float32)
    m = (np.array(brow) > 127)[..., None].astype(np.float32)
    ov = ov * (1 - m * 0.5) + np.array([255, 0, 0], np.float32) * m * 0.5

    ax.imshow(face_crop(Image.fromarray(ov.astype(np.uint8)), pad=1.6, size=460))
    ax.set_title(name[6:16], fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from PIL import ImageFilter

BROW_DIR = BASE / "outputs/salon_brow"
BROW_DIR.mkdir(parents=True, exist_ok=True)

pairs = {}
print(f"{'file':<16}{'눈썹생성':>10}{'눈썹보존':>10}{'변화':>9}")

for path in s_files:
    name = path.stem
    img_r = Image.open(SALON_OUT / f"src_{name}.png").convert("RGB")
    gen = Image.open(SALON_OUT / f"gen_{name}.png").convert("RGB")

    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]
    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)
    brow_mask = build_brow_mask(img_r)

    b = compose.color_transfer(gen, img_r, gen_mask)
    base = compose.recompose_with_hair(img_r, b.resize(img_r.size), face_mask, hair_mask)

    # 헤어를 되돌리는 것과 같은 방식. 경계 블러도 헤어와 같은 3 을 쓴다
    kept = Image.composite(img_r, base, brow_mask.filter(ImageFilter.GaussianBlur(3)))
    kept.save(BROW_DIR / f"c3_{name}.png")

    pairs[name] = (img_r, base, kept, brow_mask)
    i0, i1 = cos_sim(img_r, base, app), cos_sim(img_r, kept, app)
    print(f"{name[6:20]:<16}{i0:>10.4f}{i1:>10.4f}{i1 - i0:>+9.4f}")


def brow_crop(img, mask, size=520, pad=0.6):
    """눈썹 마스크 범위를 잡고 아래로 더 늘려 눈과의 관계까지 본다."""
    ys, xs = np.where(np.array(mask) > 127)
    h = ys.max() - ys.min()
    c = img.crop((
        xs.min(), max(0, int(ys.min() - h * pad)),
        xs.max(), min(img.height, int(ys.max() + h * pad * 2)),
    ))
    return c.resize((size, int(size * c.height / c.width)), Image.LANCZOS)


LABELS_B = ("원본", "눈썹 생성", "눈썹 보존")
fig, axes = plt.subplots(5, 3, figsize=(16, 20))
for r, name in enumerate(S_ORDER):
    img_r, base, kept, bm = pairs[name]
    for c, (lab, im) in enumerate(zip(LABELS_B, (img_r, base, kept))):
        axes[r, c].imshow(brow_crop(im, bm))
        axes[r, c].set_title(f"{lab}  {name[6:14]}", fontsize=9)
        axes[r, c].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
LABELS_F = ("원본", "눈썹 생성", "눈썹 보존")

fig, axes = plt.subplots(5, 3, figsize=(16, 24))
for r, name in enumerate(S_ORDER):
    img_r, base, kept, _ = pairs[name]
    for c, (lab, im) in enumerate(zip(LABELS_F, (img_r, base, kept))):
        axes[r, c].imshow(im)
        axes[r, c].set_title(f"{lab}  {name[6:16]}", fontsize=9)
        axes[r, c].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
BROW_CF = Path("/content/cf_brow")

%cd /content/CodeFormer
!python inference_codeformer.py -w 0.7 -i {BROW_DIR} -o {BROW_CF} 2>&1 | tail -1
%cd /content/SalonCutAI/backend

BROW_CF_DIR = BROW_CF / "final_results"

print(f"{'file':<16}{'보존':>9}{'보존+복원':>11}{'변화':>9}")
for name in S_ORDER:
    img_r, base, kept, bm = pairs[name]
    cf = Image.open(BROW_CF_DIR / f"c3_{name}.png").convert("RGB").resize(img_r.size)
    i0, i1 = cos_sim(img_r, kept, app), cos_sim(img_r, cf, app)
    print(f"{name[6:20]:<16}{i0:>9.4f}{i1:>11.4f}{i1 - i0:>+9.4f}")

LABELS_C = ("원본", "눈썹 생성", "눈썹 보존", "보존+복원")
fig, axes = plt.subplots(5, 4, figsize=(20, 20))
for r, name in enumerate(S_ORDER):
    img_r, base, kept, bm = pairs[name]
    cf = Image.open(BROW_CF_DIR / f"c3_{name}.png").convert("RGB").resize(img_r.size)
    for c, (lab, im) in enumerate(zip(LABELS_C, (img_r, base, kept, cf))):
        axes[r, c].imshow(brow_crop(im, bm))
        axes[r, c].set_title(f"{lab}  {name[6:14]}", fontsize=9)
        axes[r, c].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
LABELS_C = ("원본", "눈썹 생성", "눈썹 보존", "보존+복원")

fig, axes = plt.subplots(5, 4, figsize=(20, 24))
for r, name in enumerate(S_ORDER):
    img_r, base, kept, _ = pairs[name]
    cf = Image.open(BROW_CF_DIR / f"c3_{name}.png").convert("RGB").resize(img_r.size)
    for c, (lab, im) in enumerate(zip(LABELS_C, (img_r, base, kept, cf))):
        axes[r, c].imshow(im)
        axes[r, c].set_title(f"{lab}  {name[6:16]}", fontsize=9)
        axes[r, c].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
STAGE_SET = ("02_long_wave_dark", "04_long_wave_black")
STAGE_LABELS = ("원본", "생성", "재합성", "눈썹 보존", "보존+복원")


def iris_crop(img, side, size=460):
    """한쪽 눈만 크게. side 0 은 화면 왼쪽, 1 은 오른쪽."""
    det = app.get(np.array(img)[:, :, ::-1])
    if not det:
        return None
    kps = det[0].kps[:2]
    kps = kps[np.argsort(kps[:, 0])]
    cx, cy = kps[side]
    half = abs(kps[1][0] - kps[0][0]) * 0.42
    box = (
        max(0, int(cx - half)), max(0, int(cy - half * 0.7)),
        min(img.width, int(cx + half)), min(img.height, int(cy + half * 0.7)),
    )
    c = img.crop(box)
    return c.resize((size, int(size * c.height / c.width)), Image.LANCZOS)


for name, side in (("salon_02_long_wave_dark", 0), ("salon_04_long_wave_black", 1)):
    img_r, base, kept, _ = pairs[name]
    gen = Image.open(SALON_OUT / f"gen_{name}.png").convert("RGB").resize(img_r.size)
    cf = Image.open(BROW_CF_DIR / f"c3_{name}.png").convert("RGB").resize(img_r.size)

    fig, axes = plt.subplots(1, 5, figsize=(24, 5))
    for ax, (lab, im) in zip(axes, zip(STAGE_LABELS, (img_r, gen, base, kept, cf))):
        cr = iris_crop(im, side)
        ax.imshow(cr if cr else Image.new("RGB", (460, 300)))
        ax.set_title(lab, fontsize=11)
        ax.axis("off")
    plt.tight_layout()
    plt.suptitle(f"{name}  {'화면 왼쪽' if side == 0 else '화면 오른쪽'} 눈", y=1.04, fontsize=13)
    plt.show()

In [ ]:
from src.ai_engine.image_gen import settings as gen_settings

SCALES = (0.5, 0.35, 0.2)
EYE_SET = (("salon_02_long_wave_dark", 0), ("salon_04_long_wave_black", 1))

IPA_DIR = BASE / "outputs/salon_ipa"
ipa = {}
ref01 = Image.open(REF / "ref-01.png").convert("RGB")

print(f"{'file':<20}{'scale':>7}{'참조':>9}{'원본':>9}")
for name, _ in EYE_SET:
    path = next(p for p in s_files if p.stem == name)

    for s in SCALES:
        # __call__ 인자로는 안 먹는다. 파이프라인에 직접 설정해야 한다
        gen_loader.get_combo3().set_ip_adapter_scale(s)

        out, img_r, _ = combo3.generate(
            Image.open(path).convert("RGB"), REF / "ref-01.png", SEED
        )

        det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
        fw = det[0].bbox[2] - det[0].bbox[0]
        face_mask = masks.build_face_mask(img_r)
        hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
        gen_mask = masks.build_gen_mask(face_mask, hair_mask)
        brow_mask = build_brow_mask(img_r)

        b = compose.color_transfer(out, img_r, gen_mask)
        c = compose.recompose_with_hair(img_r, b.resize(img_r.size), face_mask, hair_mask)
        kept = Image.composite(img_r, c, brow_mask.filter(ImageFilter.GaussianBlur(3)))

        d = IPA_DIR / f"s{s}"
        d.mkdir(parents=True, exist_ok=True)
        kept.save(d / f"c3_{name}.png")
        ipa[(name, s)] = (img_r, kept)

        print(
            f"{name[6:24]:<20}{s:>7.2f}"
            f"{cos_sim(ref01, kept, app):>9.4f}{cos_sim(img_r, kept, app):>9.4f}"
        )

gen_loader.get_combo3().set_ip_adapter_scale(0.5)      # 실동작 기본값으로 원복


for name, side in EYE_SET:
    img_r = ipa[(name, SCALES[0])][0]
    cols = [("원본", img_r)] + [(f"scale {s}", ipa[(name, s)][1]) for s in SCALES]

    fig, axes = plt.subplots(1, len(cols), figsize=(5 * len(cols), 5))
    for ax, (lab, im) in zip(axes, cols):
        cr = iris_crop(im, side)
        ax.imshow(cr if cr else Image.new("RGB", (460, 300)))
        ax.set_title(lab, fontsize=11)
        ax.axis("off")
    plt.tight_layout()
    plt.suptitle(name, y=1.04, fontsize=13)
    plt.show()

In [ ]:
import inspect

pipe = gen_loader.get_combo3()

sig = inspect.signature(pipe.__call__)
print("ip_adapter_scale 인자:", "ip_adapter_scale" in sig.parameters)
print("VAR_KEYWORD 있음:", any(p.kind == p.VAR_KEYWORD for p in sig.parameters.values()))
print("set_ip_adapter_scale:", hasattr(pipe, "set_ip_adapter_scale"))

for n, m in pipe.unet.named_modules():
    if hasattr(m, "scale") and "attn2" in n and "processor" in n:
        print("현재 scale:", n, m.scale)
        break

In [ ]:
import time

import torch
from basicsr.utils import img2tensor, tensor2img
from basicsr.utils.registry import ARCH_REGISTRY
from facexlib.utils.face_restoration_helper import FaceRestoreHelper
from torchvision.transforms.functional import normalize

CF_CKPT = Path("/content/CodeFormer/weights/CodeFormer/codeformer.pth")

# --- 모델을 CPU 로 올린다 ---
net_cpu = ARCH_REGISTRY.get("CodeFormer")(
    dim_embd=512, codebook_size=1024, n_head=8, n_layers=9,
    connect_list=["32", "64", "128", "256"],
).to("cpu")
net_cpu.load_state_dict(torch.load(CF_CKPT, map_location="cpu")["params_ema"])
net_cpu.eval()
print("CodeFormer CPU 로드 완료")


def restore_512_cpu(face_bgr, w=0.7):
    """정렬된 512 얼굴 한 장의 순수 복원 시간. 검출·되붙이기 제외."""
    t = img2tensor(face_bgr / 255.0, bgr2rgb=True, float32=True)
    normalize(t, (0.5,) * 3, (0.5,) * 3, inplace=True)
    t = t.unsqueeze(0).to("cpu")
    with torch.no_grad():
        out = net_cpu(t, w=w, adain=True)[0]
    return tensor2img(out, rgb2bgr=True, min_max=(-1, 1))


# --- 1. 순수 복원만 (512 정렬 얼굴) ---
import cv2

sample = cv2.imread(str(NOHF_CF / "final_results" / "c3_salon_01_long_wave_brown.png"))
sample512 = cv2.resize(sample, (512, 512))

restore_512_cpu(sample512)   # 워밍업 (첫 호출은 그래프 초기화로 느림)

times = []
for _ in range(5):
    t = time.time()
    restore_512_cpu(sample512)
    times.append(time.time() - t)
print(f"\n순수 복원 512  평균 {sum(times)/len(times):.2f}s  (min {min(times):.2f} max {max(times):.2f})")


# --- 2. 검출·정렬·되붙이기 포함 (실서버 시나리오) ---
helper = FaceRestoreHelper(
    upscale_factor=1, face_size=512, crop_ratio=(1, 1),
    det_model="retinaface_resnet50", save_ext="png", use_parse=True, device="cpu",
)

full_times = []
for name in S_ORDER:
    img = cv2.imread(str(NOHF / f"c3_{name}.png"))
    t = time.time()

    helper.clean_all()
    helper.read_image(img)
    helper.get_face_landmarks_5(only_center_face=False, resize=640, eye_dist_threshold=5)
    helper.align_warp_face()

    for cropped in helper.cropped_faces:
        restored = restore_512_cpu(cropped)
        helper.add_restored_face(restored)

    helper.get_inverse_affine(None)
    helper.paste_faces_to_input_image()

    full_times.append(time.time() - t)
    print(f"{name[6:24]:<20}{full_times[-1]:>7.2f}s")

print(f"\n전체 파이프 평균 {sum(full_times)/len(full_times):.2f}s")

In [ ]:
from getpass import getpass

from openai import OpenAI

client = OpenAI(api_key=getpass("OpenAI API key: "))
print("클라이언트 준비 완료")

In [ ]:
import base64
import io

GPT_DIR = BASE / "outputs/salon_gpt"
GPT_DIR.mkdir(parents=True, exist_ok=True)


def to_png_bytes(img, name):
    """PIL 이미지를 named PNG 파일객체로. API 가 파일명을 요구한다."""
    buf = io.BytesIO()
    img.save(buf, "PNG")
    buf.seek(0)
    buf.name = name
    return buf


def gpt_fix_eyes(gen_img, prompt, size="1024x1024"):
    """생성 결과 1장만 주고 눈을 자연스럽게 다시 그린다. 원본 참조 없음."""
    res = client.images.edit(
        model="gpt-image-1",
        image=to_png_bytes(gen_img, "gen.png"),
        prompt=prompt,
        input_fidelity="high",
        size=size,
        n=1,
    )
    b64 = res.data[0].b64_json
    return Image.open(io.BytesIO(base64.b64decode(b64))).convert("RGB")


PROMPT = (
    "Photorealistic portrait retouch. Keep the person's identity, face shape, "
    "skin tone, makeup, hairstyle, clothing, pose, and background exactly the "
    "same. The only change: make the eyes look natural and well-focused. "
    "Fix the irises so they sit naturally within the eyes with correct, even "
    "pupil size and a normal amount of visible sclera — not too much white, "
    "not staring. Keep both eyes symmetric and consistent with the head angle. "
    "Do not change anything except the eyes."
)
print("함수·프롬프트 준비 완료")

In [ ]:
EYE_SET = ("salon_02_long_wave_dark", "salon_04_long_wave_black")
gpt_out = {}

for name in EYE_SET:
    img_r = pairs[name][0]
    gen = Image.open(SALON_OUT / f"gen_{name}.png").convert("RGB").resize(img_r.size)

    fixed = gpt_fix_eyes(gen, PROMPT)              # 원본 안 넘김
    fixed.save(GPT_DIR / f"gpt_{name}.png")
    gpt_out[name] = (img_r, gen, fixed)
    print(f"{name[6:24]:<24} 완료")

for name in EYE_SET:
    img_r, gen, fixed = gpt_out[name]
    print(f"{name[6:20]:<16} GPT-생성 {cos_sim(gen, fixed, app):.4f}   GPT-원본 {cos_sim(img_r, fixed, app):.4f}")

for name in EYE_SET:
    img_r, gen, fixed = gpt_out[name]
    side = 0 if "02" in name else 1
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (lab, im) in zip(axes, (("원본", img_r), ("생성", gen), ("GPT 보정", fixed))):
        cr = iris_crop(im, side)
        ax.imshow(cr if cr else Image.new("RGB", (460, 300)))
        ax.set_title(lab, fontsize=11)
        ax.axis("off")
    plt.tight_layout()
    plt.suptitle(name, y=1.04, fontsize=13)
    plt.show()

for name in EYE_SET:
    img_r, gen, fixed = gpt_out[name]
    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    for ax, (lab, im) in zip(axes, (("원본", img_r), ("생성", gen), ("GPT 보정", fixed))):
        ax.imshow(im)
        ax.set_title(lab, fontsize=11)
        ax.axis("off")
    plt.tight_layout()
    plt.suptitle(f"{name} — 전체", y=1.02, fontsize=13)
    plt.show()

In [ ]:
for name in EYE_SET:
    img_r, gen, fixed = gpt_out[name]
    print(f"{name[6:20]:<16} 생성-원본 {cos_sim(img_r, gen, app):.4f}   GPT-원본 {cos_sim(img_r, fixed, app):.4f}   GPT-생성 {cos_sim(gen, fixed, app):.4f}")

In [ ]:
CROP_TEST = ("salon_02_long_wave_dark", "salon_04_long_wave_black", "salon_05_short_bob_brown")


def face_box_padded(img, pad):
    """얼굴 박스를 pad 배율로 확장한 정사각 크롭 좌표. 이미지 밖으로 안 나가게 클램프."""
    det = gen_loader.get_face_app().get(np.array(img)[:, :, ::-1])
    if not det:
        return None
    x1, y1, x2, y2 = det[0].bbox
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    half = max(x2 - x1, y2 - y1) / 2 * pad
    L = min(cx, cy, img.width - cx, img.height - cy, half)   # 정사각 유지하며 클램프
    return (int(cx - L), int(cy - L), int(cx + L), int(cy + L))


print(f"{'file':<20}{'원본얼굴':>9}{'pad':>6}{'크롭크기':>9}{'1024얼굴':>10}{'추정눈폭':>10}")
for name in CROP_TEST:
    src = Image.open(SALON_OUT / f"src_{name}.png").convert("RGB")   # 1024 원본
    det = gen_loader.get_face_app().get(np.array(src)[:, :, ::-1])
    fw0 = det[0].bbox[2] - det[0].bbox[0]

    for pad in (1.3, 1.6, 2.0):
        box = face_box_padded(src, pad)
        if box is None:
            continue
        crop_size = box[2] - box[0]
        # 크롭을 1024로 키우면 얼굴 폭이 얼마가 되나
        fw_1024 = fw0 * 1024 / crop_size
        eye_est = fw_1024 * 0.35   # 눈 폭은 대략 얼굴 폭의 35%
        print(f"{name[6:20]:<20}{fw0:>9.0f}{pad:>6.1f}{crop_size:>9}{fw_1024:>10.0f}{eye_est:>10.0f}")

In [ ]:
CROP_PAD = 1.6
CROP_DIR = BASE / "outputs/salon_crop"
CROP_DIR.mkdir(parents=True, exist_ok=True)

gen_loader.get_combo3().set_ip_adapter_scale(0.5)   # 실동작 기본값 명시

crop_out = {}
for name in CROP_TEST:
    src = Image.open(SALON_OUT / f"src_{name}.png").convert("RGB")

    box = face_box_padded(src, CROP_PAD)
    face_crop_img = src.crop(box)
    cw, ch = face_crop_img.size

    # 크롭을 1024 기준으로 생성 (combo3 내부에서 8의 배수 정렬)
    out, img_r, _ = combo3.generate(face_crop_img, REF / "ref-01.png", SEED)

    # 생성 결과를 크롭 원래 크기로 되돌린다
    out_resized = out.resize((cw, ch))

    # --- 크롭 영역 안에서 후처리 (재합성·눈썹보존) ---
    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]
    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)
    brow_mask = build_brow_mask(img_r)

    b = compose.color_transfer(out, img_r, gen_mask)
    c = compose.recompose_with_hair(img_r, b.resize(img_r.size), face_mask, hair_mask)
    kept = Image.composite(img_r, c, brow_mask.filter(ImageFilter.GaussianBlur(3)))

    # 후처리 완료본을 크롭 크기로 되돌려 원본에 붙인다
    kept_resized = kept.resize((cw, ch))
    final = src.copy()
    final.paste(kept_resized, (box[0], box[1]))

    final.save(CROP_DIR / f"c3_{name}.png")
    crop_out[name] = (src, face_crop_img, kept, final, box)
    print(f"{name[6:24]:<24} 크롭 {cw}x{ch} → 생성 → 원위치")

print("완료")

In [ ]:
# 눈 확대: 기존 vs 크롭
for name in CROP_TEST:
    src, crop_img, kept, final, box = crop_out[name]
    base = Image.open(NOHF / f"c3_{name}.png").convert("RGB")   # 기존(크롭 안 함)
    side = 0 if "02" in name else 1

    cols = (("원본", src), ("기존", base), ("크롭", final))
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (lab, im) in zip(axes, cols):
        cr = iris_crop(im, side)
        ax.imshow(cr if cr else Image.new("RGB", (460, 300)))
        ax.set_title(lab, fontsize=11)
        ax.axis("off")
    plt.tight_layout()
    plt.suptitle(f"{name} — 눈", y=1.04, fontsize=13)
    plt.show()

# 전체: 경계 이음매 확인
for name in CROP_TEST:
    src, crop_img, kept, final, box = crop_out[name]
    base = Image.open(NOHF / f"c3_{name}.png").convert("RGB")

    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    for ax, (lab, im) in zip(axes, (("원본", src), ("기존", base), ("크롭", final))):
        ax.imshow(im)
        ax.set_title(lab, fontsize=11)
        ax.axis("off")
    plt.tight_layout()
    plt.suptitle(f"{name} — 전체", y=1.02, fontsize=13)
    plt.show()

# 정체성 수치
print(f"{'file':<16}{'기존-원본':>10}{'크롭-원본':>10}")
for name in CROP_TEST:
    src, crop_img, kept, final, box = crop_out[name]
    base = Image.open(NOHF / f"c3_{name}.png").convert("RGB")
    print(f"{name[6:20]:<16}{cos_sim(src, base, app):>10.4f}{cos_sim(src, final, app):>10.4f}")

In [ ]:
import cv2

CROP_CF_OUT = Path("/content/cf_crop")

%cd /content/CodeFormer
!python inference_codeformer.py -w 0.7 -i {CROP_DIR} -o {CROP_CF_OUT} 2>&1 | tail -1
%cd /content/SalonCutAI/backend

CROP_CF_DIR = CROP_CF_OUT / "final_results"

print(f"{'file':<16}{'기존완성-원본':>13}{'크롭완성-원본':>13}")
for name in CROP_TEST:
    src = crop_out[name][0]
    base_final = Image.open(NOHF_CF / "final_results" / f"c3_{name}.png").convert("RGB").resize(src.size)
    crop_final = Image.open(CROP_CF_DIR / f"c3_{name}.png").convert("RGB").resize(src.size)
    print(f"{name[6:20]:<16}{cos_sim(src, base_final, app):>13.4f}{cos_sim(src, crop_final, app):>13.4f}")

for name in CROP_TEST:
    src = crop_out[name][0]
    base_final = Image.open(NOHF_CF / "final_results" / f"c3_{name}.png").convert("RGB").resize(src.size)
    crop_final = Image.open(CROP_CF_DIR / f"c3_{name}.png").convert("RGB").resize(src.size)

    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    for ax, (lab, im) in zip(axes, (("원본", src), ("기존 완성", base_final), ("크롭 완성", crop_final))):
        ax.imshow(im)
        ax.set_title(lab, fontsize=11)
        ax.axis("off")
    plt.tight_layout()
    plt.suptitle(f"{name} — 복원까지", y=1.02, fontsize=13)
    plt.show()

In [ ]:
CROP_REF_TEST = ("salon_02_long_wave_dark", "salon_04_long_wave_black")
CROP_REFS = ("ref-02", "ref-09")

gen_loader.get_combo3().set_ip_adapter_scale(0.5)

crop_ref_out = {}
for r in CROP_REFS:
    rdir = BASE / f"outputs/salon_crop_{r}"
    rdir.mkdir(parents=True, exist_ok=True)

    for name in CROP_REF_TEST:
        src = Image.open(SALON_OUT / f"src_{name}.png").convert("RGB")
        box = face_box_padded(src, CROP_PAD)
        face_crop_img = src.crop(box)
        cw, ch = face_crop_img.size

        out, img_r, _ = combo3.generate(face_crop_img, REF / f"{r}.png", SEED)

        det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
        fw = det[0].bbox[2] - det[0].bbox[0]
        face_mask = masks.build_face_mask(img_r)
        hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
        gen_mask = masks.build_gen_mask(face_mask, hair_mask)
        brow_mask = build_brow_mask(img_r)

        b = compose.color_transfer(out, img_r, gen_mask)
        c = compose.recompose_with_hair(img_r, b.resize(img_r.size), face_mask, hair_mask)
        kept = Image.composite(img_r, c, brow_mask.filter(ImageFilter.GaussianBlur(3)))

        final = src.copy()
        final.paste(kept.resize((cw, ch)), (box[0], box[1]))
        final.save(rdir / f"c3_{name}.png")
        crop_ref_out[(r, name)] = (src, final, box)
    print(f"{r} 완료")

# 눈 확대: 참조별로, 크롭 전(기존 ref) vs 크롭
for name in CROP_REF_TEST:
    side = 0 if "02" in name else 1
    for r in CROP_REFS:
        src, final, box = crop_ref_out[(r, name)]
        # 크롭 전 기존 결과 (해당 참조)
        old_dir = REF_DIRS[r] if r in REF_DIRS else None
        old = Image.open(old_dir / f"c3_{name}.png").convert("RGB") if old_dir and (old_dir / f"c3_{name}.png").exists() else None

        cols = [("원본", src)]
        if old is not None:
            cols.append((f"{r} 크롭전", old.resize(src.size)))
        cols.append((f"{r} 크롭", final))

        fig, axes = plt.subplots(1, len(cols), figsize=(5 * len(cols), 5))
        for ax, (lab, im) in zip(axes, cols):
            cr = iris_crop(im, side)
            ax.imshow(cr if cr else Image.new("RGB", (460, 300)))
            ax.set_title(lab, fontsize=11)
            ax.axis("off")
        plt.tight_layout()
        plt.suptitle(f"{name} — {r}", y=1.04, fontsize=13)
        plt.show()

In [ ]:
for name in CROP_REF_TEST:
    for r in CROP_REFS:
        src, final, box = crop_ref_out[(r, name)]
        old_dir = REF_DIRS.get(r)
        old = (
            Image.open(old_dir / f"c3_{name}.png").convert("RGB").resize(src.size)
            if old_dir and (old_dir / f"c3_{name}.png").exists() else None
        )

        cols = [("원본", src)]
        if old is not None:
            cols.append((f"{r} 크롭전", old))
        cols.append((f"{r} 크롭", final))

        fig, axes = plt.subplots(1, len(cols), figsize=(5 * len(cols), 6))
        for ax, (lab, im) in zip(axes, cols):
            ax.imshow(im)
            ax.set_title(lab, fontsize=11)
            ax.axis("off")
        plt.tight_layout()
        plt.suptitle(f"{name} — {r} 전체", y=1.02, fontsize=13)
        plt.show()

In [ ]:
EYE_CONNS = (
    vision.FaceLandmarksConnections.FACE_LANDMARKS_LEFT_EYE,
    vision.FaceLandmarksConnections.FACE_LANDMARKS_RIGHT_EYE,
)


def build_eye_mask(img, expand=1.35):
    """양 눈 영역. 눈꺼풀 바깥까지 덮도록 볼록껍질을 중심에서 확장한다."""
    w, h = img.size
    res = gen_loader.get_landmarker().detect(masks._to_mp_image(img))
    if not res.face_landmarks:
        return None
    lm = res.face_landmarks[0]

    mask = np.zeros((h, w), np.uint8)
    for conns in EYE_CONNS:
        idx = sorted({c.start for c in conns} | {c.end for c in conns})
        pts = np.array([[lm[i].x * w, lm[i].y * h] for i in idx], np.float32)
        c = pts.mean(axis=0)
        pts = ((pts - c) * expand + c).astype(np.int32)   # 경계를 피부 쪽으로
        cv2.fillConvexPoly(mask, cv2.convexHull(pts), 255)

    return Image.fromarray(mask)


EYEMIX_DIR = BASE / "outputs/salon_eyemix"
EYEMIX_DIR.mkdir(parents=True, exist_ok=True)

eyemix = {}
print(f"{'file':<24}{'기존':>9}{'크롭':>9}{'눈만':>9}")
for r in CROP_REFS:
    for name in CROP_REF_TEST:
        src, crop_final, box = crop_ref_out[(r, name)]
        base = Image.open(REF_DIRS[r] / f"c3_{name}.png").convert("RGB").resize(src.size)

        eye_mask = build_eye_mask(src)
        mixed = Image.composite(
            crop_final, base, eye_mask.filter(ImageFilter.GaussianBlur(6))
        )

        d = EYEMIX_DIR / r
        d.mkdir(parents=True, exist_ok=True)
        mixed.save(d / f"c3_{name}.png")
        eyemix[(r, name)] = (src, base, crop_final, mixed)

        print(
            f"{r} {name[6:20]:<18}"
            f"{cos_sim(src, base, app):>9.4f}{cos_sim(src, crop_final, app):>9.4f}"
            f"{cos_sim(src, mixed, app):>9.4f}"
        )

In [ ]:
for r in CROP_REFS:
    for name in CROP_REF_TEST:
        src, base, crop_final, mixed = eyemix[(r, name)]
        side = 0 if "02" in name else 1
        cols = (("원본", src), ("기존", base), ("크롭", crop_final), ("눈만", mixed))

        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        for ax, (lab, im) in zip(axes, cols):
            cr = iris_crop(im, side)
            ax.imshow(cr if cr else Image.new("RGB", (460, 300)))
            ax.set_title(lab, fontsize=11)
            ax.axis("off")
        plt.tight_layout()
        plt.suptitle(f"{name} — {r} 눈", y=1.04, fontsize=13)
        plt.show()

for r in CROP_REFS:
    for name in CROP_REF_TEST:
        src, base, crop_final, mixed = eyemix[(r, name)]
        fig, axes = plt.subplots(1, 4, figsize=(20, 6))
        for ax, (lab, im) in zip(axes, (("원본", src), ("기존", base), ("크롭", crop_final), ("눈만", mixed))):
            ax.imshow(im)
            ax.set_title(lab, fontsize=11)
            ax.axis("off")
        plt.tight_layout()
        plt.suptitle(f"{name} — {r} 전체", y=1.02, fontsize=13)
        plt.show()

In [ ]:
BASE_DIR = BASE / "outputs/salon_base_v2"
BASE_DIR.mkdir(parents=True, exist_ok=True)

gen_loader.get_combo3().set_ip_adapter_scale(0.5)

eyemix2 = {}
print(f"{'file':<24}{'베이스':>9}{'크롭':>9}{'눈만':>9}")
for r in CROP_REFS:
    d = BASE_DIR / r
    d.mkdir(parents=True, exist_ok=True)

    for name in CROP_REF_TEST:
        src = Image.open(SALON_OUT / f"src_{name}.png").convert("RGB")

        # 크롭 없이, 확정 파이프라인으로 베이스 생성
        out, img_r, _ = combo3.generate(src, REF / f"{r}.png", SEED)
        det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
        fw = det[0].bbox[2] - det[0].bbox[0]
        face_mask = masks.build_face_mask(img_r)
        hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
        gen_mask = masks.build_gen_mask(face_mask, hair_mask)
        brow_mask = build_brow_mask(img_r)

        b = compose.color_transfer(out, img_r, gen_mask)
        c = compose.recompose_with_hair(img_r, b.resize(img_r.size), face_mask, hair_mask)
        base = Image.composite(img_r, c, brow_mask.filter(ImageFilter.GaussianBlur(3)))
        base.save(d / f"c3_{name}.png")

        # 크롭 결과에서 눈만
        crop_final = crop_ref_out[(r, name)][1]
        eye_mask = build_eye_mask(src)
        mixed = Image.composite(
            crop_final.resize(base.size), base,
            eye_mask.resize(base.size).filter(ImageFilter.GaussianBlur(6)),
        )
        eyemix2[(r, name)] = (src, base, crop_final, mixed)

        print(
            f"{r} {name[6:20]:<18}"
            f"{cos_sim(src, base, app):>9.4f}{cos_sim(src, crop_final, app):>9.4f}"
            f"{cos_sim(src, mixed, app):>9.4f}"
        )

# 전체 비교
for r in CROP_REFS:
    for name in CROP_REF_TEST:
        src, base, crop_final, mixed = eyemix2[(r, name)]
        fig, axes = plt.subplots(1, 4, figsize=(20, 6))
        for ax, (lab, im) in zip(axes, (("원본", src), ("베이스", base), ("크롭", crop_final), ("눈만", mixed))):
            ax.imshow(im)
            ax.set_title(lab, fontsize=11)
            ax.axis("off")
        plt.tight_layout()
        plt.suptitle(f"{name} — {r} 전체", y=1.02, fontsize=13)
        plt.show()

In [ ]:
CROP_CF2 = Path("/content/cf_crop_v2")

%cd /content/CodeFormer
for r in CROP_REFS:
    src_dir = BASE / f"outputs/salon_crop_{r}"
    out_dir = CROP_CF2 / r
    !python inference_codeformer.py -w 0.7 -i {src_dir} -o {out_dir} 2>&1 | tail -1
%cd /content/SalonCutAI/backend

final_cmp = {}
print(f"{'file':<24}{'베이스+복원':>12}{'크롭+복원':>12}")
for r in CROP_REFS:
    for name in CROP_REF_TEST:
        src = Image.open(SALON_OUT / f"src_{name}.png").convert("RGB")
        bf = Image.open(BASE_CF / r / "final_results" / f"c3_{name}.png").convert("RGB").resize(src.size)
        cf = Image.open(CROP_CF2 / r / "final_results" / f"c3_{name}.png").convert("RGB").resize(src.size)
        final_cmp[(r, name)] = (src, bf, cf)
        print(f"{r} {name[6:20]:<18}{cos_sim(src, bf, app):>12.4f}{cos_sim(src, cf, app):>12.4f}")

for r in CROP_REFS:
    for name in CROP_REF_TEST:
        src, bf, cf = final_cmp[(r, name)]
        fig, axes = plt.subplots(1, 3, figsize=(15, 6))
        for ax, (lab, im) in zip(axes, (("원본", src), ("크롭 없이", bf), ("크롭 적용", cf))):
            ax.imshow(im)
            ax.set_title(lab, fontsize=11)
            ax.axis("off")
        plt.tight_layout()
        plt.suptitle(f"{name} — {r} 최종(복원까지)", y=1.02, fontsize=13)
        plt.show()

In [ ]:
for r in CROP_REFS:
    for name in CROP_REF_TEST:
        src, bf, cf = final_cmp[(r, name)]
        side = 0 if "02" in name else 1
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        for ax, (lab, im) in zip(axes, (("원본", src), ("크롭 없이", bf), ("얼굴 크롭", cf))):
            cr = iris_crop(im, side)
            ax.imshow(cr if cr else Image.new("RGB", (460, 300)))
            ax.set_title(lab, fontsize=11)
            ax.axis("off")
        plt.tight_layout()
        plt.suptitle(f"{name} — {r} 최종 눈", y=1.04, fontsize=13)
        plt.show()

In [ ]:
BASE_ALL = BASE / "outputs/salon_base_all"
BASE_ALL.mkdir(parents=True, exist_ok=True)

gen_loader.get_combo3().set_ip_adapter_scale(0.5)

base_all = {}
for path in s_files:
    name = path.stem
    src = Image.open(SALON_OUT / f"src_{name}.png").convert("RGB")

    out, img_r, _ = combo3.generate(src, REF / "ref-01.png", SEED)
    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]
    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)
    brow_mask = build_brow_mask(img_r)

    b = compose.color_transfer(out, img_r, gen_mask)
    c = compose.recompose_with_hair(img_r, b.resize(img_r.size), face_mask, hair_mask)
    kept = Image.composite(img_r, c, brow_mask.filter(ImageFilter.GaussianBlur(3)))
    kept.save(BASE_ALL / f"c3_{name}.png")
    base_all[name] = (src, kept)
    print(f"{name[6:24]:<24} 완료")

fig, axes = plt.subplots(5, 2, figsize=(11, 22))
for r, name in enumerate(S_ORDER):
    src, kept = base_all[name]
    side = 0 if name in ("salon_02_long_wave_dark",) else 1
    for c, (lab, im) in enumerate((("원본", src), ("크롭 없이", kept))):
        cr = iris_crop(im, side)
        axes[r, c].imshow(cr if cr else Image.new("RGB", (460, 300)))
        axes[r, c].set_title(f"{lab}  {name[6:16]}", fontsize=9)
        axes[r, c].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
BASE_ALL_CF = Path("/content/cf_base_all")

%cd /content/CodeFormer
!python inference_codeformer.py -w 0.7 -i {BASE_ALL} -o {BASE_ALL_CF} 2>&1 | tail -1
%cd /content/SalonCutAI/backend

CF_ALL = BASE_ALL_CF / "final_results"

print(f"{'file':<20}{'복원전':>9}{'복원후':>9}{'질감전':>9}{'질감후':>9}")
cmp_all = {}
for name in S_ORDER:
    src, pre = base_all[name]
    post = Image.open(CF_ALL / f"c3_{name}.png").convert("RGB").resize(src.size)
    cmp_all[name] = (src, pre, post)
    print(
        f"{name[6:20]:<20}"
        f"{cos_sim(src, pre, app):>9.4f}{cos_sim(src, post, app):>9.4f}"
        f"{lap_var(pre, app):>9.1f}{lap_var(post, app):>9.1f}"
    )

# 눈 확대
fig, axes = plt.subplots(5, 3, figsize=(16, 22))
for r, name in enumerate(S_ORDER):
    src, pre, post = cmp_all[name]
    side = 0 if name == "salon_02_long_wave_dark" else 1
    for c, (lab, im) in enumerate((("원본", src), ("복원 전", pre), ("복원 후", post))):
        cr = iris_crop(im, side)
        axes[r, c].imshow(cr if cr else Image.new("RGB", (460, 300)))
        axes[r, c].set_title(f"{lab}  {name[6:16]}", fontsize=9)
        axes[r, c].axis("off")
plt.tight_layout()
plt.show()

# 전체
for name in S_ORDER:
    src, pre, post = cmp_all[name]
    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    for ax, (lab, im) in zip(axes, (("원본", src), ("복원 전", pre), ("복원 후", post))):
        ax.imshow(im)
        ax.set_title(lab, fontsize=11)
        ax.axis("off")
    plt.tight_layout()
    plt.suptitle(f"{name} — 복원 전후", y=1.02, fontsize=13)
    plt.show()

In [ ]:
FINAL_DIR = BASE / "outputs/salon_final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

final_all = {}
print(f"{'file':<20}{'복원후':>9}{'색정합후':>10}{'마스크밖ΔE':>12}")
for name in S_ORDER:
    src, pre, post = cmp_all[name]

    det = gen_loader.get_face_app().get(np.array(src)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]
    face_mask = masks.build_face_mask(src)
    hair_mask = masks.build_hair_mask(src, dilate=max(1, int(fw * DILATE_RATIO)))
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    shifted = compose.color_transfer(post, src, gen_mask)
    fin = Image.composite(shifted, post, gen_mask.convert("L"))
    fin.save(FINAL_DIR / f"c3_{name}.png")
    final_all[name] = (src, post, fin)

    m = np.array(gen_mask) > 127
    de = delta_e(post, fin)
    print(
        f"{name[6:20]:<20}{cos_sim(src, post, app):>9.4f}"
        f"{cos_sim(src, fin, app):>10.4f}{de[~m].mean():>12.2f}"
    )

for name in S_ORDER:
    src, post, fin = final_all[name]
    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    for ax, (lab, im) in zip(axes, (("원본", src), ("복원 후", post), ("색정합까지"), )):
        pass
    plt.close(fig)

for name in S_ORDER:
    src, post, fin = final_all[name]
    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    for ax, (lab, im) in zip(axes, (("원본", src), ("복원 후", post), ("색정합까지", fin))):
        ax.imshow(im)
        ax.set_title(lab, fontsize=11)
        ax.axis("off")
    plt.tight_layout()
    plt.suptitle(f"{name} — 최종", y=1.02, fontsize=13)
    plt.show()

In [ ]:
for name in S_ORDER:
    src, post, fin = final_all[name]
    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    for ax, (lab, im) in zip(axes, (("원본", src), ("복원 후", post), ("색정합까지", fin))):
        ax.imshow(im)
        ax.set_title(lab, fontsize=11)
        ax.axis("off")
    plt.tight_layout()
    plt.suptitle(f"{name} — 최종", y=1.02, fontsize=13)
    plt.show()